# CheXpert label-efficiency study — local Jupyter

Open **`chexpert_ssl.ipynb` in the project root** using Jupyter Notebook, JupyterLab, or VS Code's Jupyter extension. The first cell locates this notebook and sets its directory as the kernel working directory. Dataset, cache, configuration and output paths are rooted there. If your kernel starts elsewhere, set `CHEXPERT_PROJECT_ROOT` to this directory before running the first cell.

This notebook owns the implementation and YAML settings. Run the definition cells, then enable the workflow stages below one at a time. **Run All with the shipped defaults starts no installation, training, or held-out evaluation.** Optional dependency installation is available below; dataset acquisition remains external.

Methods: SimCLR, MoCo v2, BYOL, NNCLR and SwAV; frozen probes and fine-tuning; scratch and pinned ImageNet baselines. The reference matrix contains 5 pretraining and 180 downstream runs. Read `docs/context.md` before changing the scientific protocol.

Save implementation edits and restart/run the definition cells. Reuse saved cohorts; changed implementations require new output and plan directories. Model outputs are research results, never clinical advice.


In [78]:
from __future__ import annotations

import os
import subprocess
import sys
from contextlib import contextmanager
from pathlib import Path


def find_project_root(start=None):
    """Resolve an explicit root or find this notebook above the working directory."""
    explicit = os.environ.get("CHEXPERT_PROJECT_ROOT")
    if explicit:
        root = Path(explicit).expanduser().resolve()
        if not (root / "chexpert_ssl.ipynb").is_file():
            raise FileNotFoundError(f"No CheXpert notebook in {root}")
        return root
    location = Path(start or Path.cwd()).resolve()
    for root in (location, *location.parents):
        if (root / "chexpert_ssl.ipynb").is_file():
            return root
    raise FileNotFoundError("Set CHEXPERT_PROJECT_ROOT to the directory containing this notebook")


PROJECT_ROOT = find_project_root()
NOTEBOOK_PATH = PROJECT_ROOT / "chexpert_ssl.ipynb"
# Jupyter uses the notebook directory as the working project root.
os.chdir(PROJECT_ROOT)


@contextmanager
def project_directory():
    previous = Path.cwd()
    os.chdir(PROJECT_ROOT)
    try:
        yield
    finally:
        os.chdir(previous)

## Optional: install dependencies into this Jupyter kernel

If dependencies are already installed, leave this disabled. Otherwise enable the next cell, run it, then restart the kernel and run all definition cells. Installation uses this kernel's Python and the project-root `pyproject.toml`; no terminal or cloud mount is needed.


In [79]:
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", ".[dev,notebook]"],
        cwd=PROJECT_ROOT,
        check=True,
    )

In [80]:
import argparse
import copy
import hashlib
import importlib.metadata
import json
import math
import multiprocessing
import random
import re
import shutil
import tempfile
import time
from collections.abc import Callable, Iterable
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from functools import lru_cache, wraps
from hashlib import sha256
from itertools import islice
from typing import Any

import numpy as np
import pandas as pd
import torch
import yaml
from PIL import Image
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    roc_curve,
)
from torch import nn
from torch.nn import functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset, RandomSampler, Sampler, TensorDataset
from torchvision import models as vision_models
from torchvision import transforms
from torchvision.utils import save_image

## YAML configuration

The bundle below retains the reference and historical profiles as named configuration entries. Names beginning `configs/` are **embedded keys**, not required files. Inheritance is resolved in memory. Change shared templates or reference overrides here; resolved JSON and provenance are saved with every run. External YAML/JSON overrides are also supported.

The reference profile uses patient split seed 1234, downstream seeds 0/1/2 and five nested budgets. `num_workers: 0` avoids notebook multiprocessing pickling failures; threaded cache decoding and device-side augmentation remain available. Dataset/output paths are relative to `PROJECT_ROOT`.

In [81]:
CONFIG_YAML = r"""
configs/common.yaml: |
  extends: models/backbones.yaml
  seed: 42
  device: auto
  optimizer: adamw
  precision: auto
  multiprocessing_context: spawn
  resume: true
  target_policy:
    Atelectasis: 1
    Cardiomegaly: 0
    Consolidation: 0
    Edema: 1
    Pleural Effusion: 0
configs/data/notebook_reference.yaml: |
  extends: prepare.yaml
  output_dir: data/processed/notebook-reference/manifests
  seed: 1234
  budget_seeds: [0, 1, 2]
  random_state: true
  budget_rounding: ceil
configs/data/prepare.yaml: |
  dataset_root: CheXpert-v1.0-small
  output_dir: data/processed/manifests
  dev_fraction: 0.1
  seed: 42
  frontal_only_downstream: true
  budgets: [0.01, 0.05, 0.10, 0.25, 1.0]
  budget_seeds: [42, 43, 44]
configs/downstream/defaults.yaml: |
  extends: ../common.yaml
  train_manifest: data/processed/manifests/downstream_train.csv
  development_manifest: data/processed/manifests/development.csv
  sampled_patients: data/processed/splits/budget_0p01_seed_42.json
  label_fraction: 0.01
  batch_size: 16
  num_workers: 0
  torch_num_threads: 4
  epochs: 50
  early_stopping_patience: 10
  weight_decay: 0.0001
  encoder_learning_rate: null
  supervised_augmentation:
    rotation: 5
    translation: [0.02, 0.02]
configs/downstream/imagenet.yaml: |
  extends: defaults.yaml
  mode: supervised
  imagenet_pretrained: true
  learning_rate: 0.0001
  output_dir: outputs/downstream/imagenet-1pct-seed42
configs/downstream/simclr_finetune.yaml: |
  extends: defaults.yaml
  mode: ssl_finetune
  ssl_method: simclr
  ssl_checkpoint: outputs/pretrain/simclr-resnet18-gpu-cache-seed42/best.pt
  output_dir: outputs/downstream/simclr-finetune-1pct-seed42
  learning_rate: 1.0e-05
configs/downstream/simclr_linear.yaml: |
  extends: defaults.yaml
  mode: ssl_linear
  ssl_method: simclr
  ssl_checkpoint: outputs/pretrain/simclr-resnet18-gpu-cache-seed42/best.pt
  output_dir: outputs/downstream/simclr-linear-1pct-seed42
  learning_rate: 0.001
configs/downstream/smoke.yaml: |
  extends: simclr_linear.yaml
  train_manifest: data/processed/smoke/downstream_train.csv
  development_manifest: data/processed/smoke/development.csv
  sampled_patients: data/processed/smoke/cohort.json
  label_fraction: 1.0
  ssl_checkpoint: outputs/smoke/simclr-resnet18-gpu-cache/best.pt
  output_dir: outputs/smoke/downstream-linear
  epochs: 1
  early_stopping_patience: 1
  batch_size: 8
  num_workers: 0
  resume: false
configs/downstream/supervised.yaml: |
  extends: defaults.yaml
  mode: supervised
  output_dir: outputs/downstream/supervised-1pct-seed42
  learning_rate: 0.0001
configs/experiments/label_efficiency.yaml: |
  name: chexpert-label-efficiency
  data_config: configs/data/prepare.yaml
  generated_dir: outputs/plans/label-efficiency
  pretrain_configs:
    simclr: configs/pretrain/simclr.yaml
    moco: configs/pretrain/moco.yaml
    byol: configs/pretrain/byol.yaml
    nnclr: configs/pretrain/nnclr.yaml
    swav: configs/pretrain/swav.yaml
  downstream_templates:
    ssl_linear: configs/downstream/simclr_linear.yaml
    ssl_finetune: configs/downstream/simclr_finetune.yaml
    scratch: configs/downstream/supervised.yaml
    imagenet: configs/downstream/imagenet.yaml
  budgets: [0.01, 0.05, 0.10, 0.25, 1.0]
  seeds: [42, 43, 44]
  splits_dir: data/processed/splits
  downstream_output_root: outputs/downstream
  evaluation_output_root: outputs/evaluation
  final_manifest: data/processed/manifests/final_validation.csv
  bootstrap_samples: 1000
configs/experiments/notebook_reference.yaml: |
  # Scientific defaults from chexpert_ssl.ipynb; legacy artifacts are separate.
  extends: label_efficiency.yaml
  name: chexpert-notebook-consolidated
  execution_order: notebook
  data_config: configs/data/notebook_reference.yaml
  generated_dir: outputs/plans/notebook-consolidated
  seeds: [0, 1, 2]
  splits_dir: data/processed/notebook-reference/splits
  pretrain_output_root: outputs/notebook-consolidated/pretrain
  downstream_output_root: outputs/notebook-consolidated/downstream
  evaluation_output_root: outputs/notebook-consolidated/evaluation
  final_manifest: data/processed/notebook-reference/manifests/final_validation.csv
  transfer_checkpoint: last.pt
  per_label_bootstrap: true
  notebook_smoke: true
  smoke_budget: 0.05
  pretrain_overrides:
    seed: 0
    manifest: data/processed/notebook-reference/manifests/pretrain_train_leakage_free.csv
    development_manifest: data/processed/notebook-reference/manifests/development.csv
    cache_path: data/processed/notebook-reference/cache/all_training_gray_128.npy
    cache_source_manifest: data/processed/notebook-reference/manifests/all_training.csv
    batch_size: 256
    learning_rate: 0.001
    notebook_schedule: true
    notebook_sampling: true
    warmup_epochs: 2
    monitor_feature_spread: true
    augmentation:
      normalization: {mean: 0.449, std: 0.226}
    ssl:
      hidden_dim: 1024
      projection_dim: 128
  method_overrides:
    moco:
      ssl: {bn_splits: 4}
    nnclr:
      ssl: {positive_as_anchor: true}
    swav:
      ssl: {prototype_freeze_epochs: 1}
  downstream_overrides:
    seed: 0
    train_manifest: data/processed/notebook-reference/manifests/downstream_train.csv
    development_manifest: data/processed/notebook-reference/manifests/development.csv
    sampled_patients: data/processed/notebook-reference/splits/budget_0p01_seed_0.json
    batch_size: 64
    epochs: 30
    early_stopping_patience: 7
    learning_rate: 0.001
    encoder_learning_rate: 0.0001
    pos_weight_max: 50.0
    schedule: cosine
    threshold_method: youden
    cached_downstream: true
    notebook_sampling: true
    training_cache_manifest: data/processed/notebook-reference/manifests/all_training.csv
    downstream_cache_path: data/processed/notebook-reference/cache/all_training_gray_128.npy
    normalization: {mean: 0.449, std: 0.226}
    supervised_augmentation:
      notebook_preprocessing: true
      normalization: {mean: 0.449, std: 0.226}
      scale: [0.8, 1.0]
      ratio: [0.9, 1.1]
      rotation: 5.0
      translation: 0.03
      brightness: 0.1
      contrast: 0.1
      blur_probability: 0.0
      noise_probability: 0.0
      noise_std: 0.0
      horizontal_flip: false
  protocol_overrides:
    ssl_linear:
      epochs: 100
      early_stopping_patience: 10
      batch_size: 512
      standardize_features: true
    scratch:
      encoder_learning_rate: 0.001
  epochs_by_budget: {0.01: 30, 0.05: 30, 0.10: 25, 0.25: 20, 1.0: 10}
configs/models/backbones.yaml: |
  architectures:
    - resnet18
    - resnet50
  encoder: resnet18
  image_size: 128
  imagenet_pretrained: false
  imagenet_weights: IMAGENET1K_V1
configs/pretrain/byol.yaml: |
  extends: defaults.yaml
  method: byol
  output_dir: outputs/pretrain/byol-resnet18-gpu-cache-seed42
  ssl:
    momentum: 0.99
configs/pretrain/defaults.yaml: |
  extends: ../common.yaml
  development_manifest: data/processed/manifests/development.csv
  schedule: cosine
  cache_log_interval: 10000
  manifest: data/processed/manifests/pretrain_train_leakage_free.csv
  cache_path: data/processed/cache/pretrain_gray_128.npy
  batch_size: 64
  num_workers: 0
  prefetch_factor: 4
  cache_workers: 16
  torch_num_threads: 4
  channels_last: true
  cudnn_benchmark: true
  allow_tf32: true
  deterministic: false
  fused_optimizer: true
  epochs: 30
  learning_rate: 0.0003
  weight_decay: 0.0001
  augmentation:
    scale:
    - 0.6
    - 1.0
    ratio:
    - 0.85
    - 1.18
    rotation: 10.0
    translation: 0.05
    brightness: 0.1
    contrast: 0.2
    blur_probability: 0.5
    blur_sigma:
    - 0.1
    - 1.5
    noise_probability: 0.5
    noise_std: 0.02
    horizontal_flip: false
  ssl:
    temperature: 0.2
    hidden_dim: 512
    projection_dim: 128
configs/pretrain/moco.yaml: |
  extends: defaults.yaml
  method: moco
  output_dir: outputs/pretrain/moco-resnet18-gpu-cache-seed42
  ssl:
    temperatures:
      moco: 0.2
    queue_size: 4096
    momentum: 0.99
configs/pretrain/nnclr.yaml: |
  extends: defaults.yaml
  method: nnclr
  output_dir: outputs/pretrain/nnclr-resnet18-gpu-cache-seed42
  ssl:
    temperatures:
      nnclr: 0.1
    queue_size: 8192
configs/pretrain/simclr.yaml: |
  extends: defaults.yaml
  method: simclr
  output_dir: outputs/pretrain/simclr-resnet18-gpu-cache-seed42
  ssl:
    temperature: 0.2
configs/pretrain/simclr_smoke.yaml: |
  extends: defaults.yaml
  manifest: data/processed/manifests/pretrain_smoke.csv
  cache_path: data/processed/cache/smoke_gray_128.npy
  output_dir: outputs/smoke/simclr-resnet18-gpu-cache
  method: simclr
  # Keep the smoke lightweight when the primary backbone/resolution changes.
  encoder: resnet18
  image_size: 128
  batch_size: 8
  num_workers: 2
  prefetch_factor: 2
  cache_workers: 4
  torch_num_threads: 2
  epochs: 1
  resume: false
  ssl:
    hidden_dim: 128
    projection_dim: 64
configs/pretrain/swav.yaml: |
  extends: defaults.yaml
  method: swav
  output_dir: outputs/pretrain/swav-resnet18-gpu-cache-seed42
  ssl:
    temperatures:
      swav: 0.1
    prototype_count: 300
    sinkhorn_epsilon: 0.05
    prototype_freeze_steps: 100
configs/prompts/chexpert_v1.yaml: |
  version: chexpert-v1
  prompts:
    Atelectasis:
      positive: ["chest radiograph with atelectasis", "x-ray showing pulmonary atelectasis"]
      negative: ["chest radiograph without atelectasis", "x-ray with no pulmonary atelectasis"]
    Cardiomegaly:
      positive: ["chest radiograph with cardiomegaly", "x-ray showing enlarged cardiac silhouette"]
      negative: ["chest radiograph without cardiomegaly", "x-ray with normal cardiac silhouette"]
    Consolidation:
      positive: ["chest radiograph with consolidation", "x-ray showing pulmonary consolidation"]
      negative: ["chest radiograph without consolidation", "x-ray with no pulmonary consolidation"]
    Edema:
      positive: ["chest radiograph with pulmonary edema", "x-ray showing edema"]
      negative: ["chest radiograph without pulmonary edema", "x-ray with no edema"]
    Pleural Effusion:
      positive: ["chest radiograph with pleural effusion", "x-ray showing pleural fluid"]
      negative: ["chest radiograph without pleural effusion", "x-ray with no pleural effusion"]
configs/vlm/finetune.yaml: |
  seed: 42
  device: auto
  mode: vlm_finetune
  checkpoint: REPLACE_WITH_PINNED_CHEXZERO_CHECKPOINT
  revision: REPLACE_WITH_PINNED_REVISION
  train_manifest: data/processed/manifests/downstream_train.csv
  development_manifest: data/processed/manifests/development.csv
  output_dir: outputs/vlm/chexzero-finetune-1pct-seed42
  label_fraction: 0.01
  batch_size: 8
  epochs: 50
  early_stopping_patience: 10
  learning_rate: 0.00001
  encoder_learning_rate: 0.000001
  weight_decay: 0.0001
  pretraining_modality: image-text VLM
  license_review: required-before-run
configs/vlm/linear_probe.yaml: |
  seed: 42
  device: auto
  mode: vlm_linear
  checkpoint: REPLACE_WITH_PINNED_CHEXZERO_CHECKPOINT
  revision: REPLACE_WITH_PINNED_REVISION
  train_manifest: data/processed/manifests/downstream_train.csv
  development_manifest: data/processed/manifests/development.csv
  output_dir: outputs/vlm/chexzero-linear-1pct-seed42
  label_fraction: 0.01
  batch_size: 16
  epochs: 50
  early_stopping_patience: 10
  learning_rate: 0.001
  weight_decay: 0.0001
  pretraining_modality: image-text VLM
  license_review: required-before-run
configs/vlm/zeroshot.yaml: |
  device: auto
  checkpoint: REPLACE_WITH_PINNED_CHEXZERO_CHECKPOINT
  revision: REPLACE_WITH_PINNED_REVISION
  manifest: data/processed/manifests/development.csv
  prompt_set: configs/prompts/chexpert_v1.yaml
  output_dir: outputs/vlm/chexzero-zeroshot-development
  batch_size: 32
  pretraining_modality: image-text VLM
  model_card_url: REPLACE_WITH_MODEL_CARD_URL
  license_review: required-before-run
"""

## Data

Patient identity, primary U-Ones/U-Zeros mapping and shared saved cohorts.

In [82]:
"""CheXpert manifests, patient-safe splits, datasets, and image transforms."""


TARGETS = ("Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion")
UNCERTAIN_AS_POSITIVE = frozenset({"Atelectasis", "Edema"})
FRONTAL_VIEWS = frozenset({"AP", "PA"})

_PATIENT_RE = re.compile(r"patient(\d+)", re.IGNORECASE)
_STUDY_RE = re.compile(r"study(\d+)", re.IGNORECASE)


def patient_identity(identifier: str) -> str:
    """Compare decimal patient IDs independently of legacy zero-padding loss."""
    value = str(identifier)
    return value.lstrip("0") or "0" if re.fullmatch(r"[0-9]+", value) else value


def read_manifest(path: str | Path, *, labeled: bool = True) -> pd.DataFrame:
    """Load prepared records without losing zero-padded patient/study identifiers."""
    frame = pd.read_csv(path, dtype={"patient_id": "string", "study_id": "string"})
    required = {"patient_id", "image_path"}
    if labeled:
        required.update(target_columns())
    missing = required.difference(frame.columns)
    if missing or frame.empty:
        raise ValueError(f"Invalid or empty manifest {path}; missing columns: {sorted(missing)}")
    if frame[list(required)].isna().any().any():
        raise ValueError(f"Manifest {path} contains missing identifiers, paths, or targets")
    if labeled and not frame[target_columns()].isin([0, 1]).all().all():
        raise ValueError(f"Manifest {path} must contain binary resolved targets")
    if labeled:
        for label in TARGETS:
            if label in frame:
                expected = frame[label].map(
                    lambda value, target=label: resolve_target(value, target)
                )
                if not expected.eq(frame[f"target_{label}"]).all():
                    raise ValueError(
                        f"Resolved {label} targets do not match the primary uncertainty policy"
                    )
    if "Path" in frame:
        expected = frame["Path"].map(
            lambda value: extract_identifier(value, _PATIENT_RE, "patient ID")
        )
        equivalent = expected.map(patient_identity).eq(frame.patient_id.map(patient_identity))
        if not equivalent.all():
            raise ValueError(f"Manifest {path} has patient IDs inconsistent with source paths")
        restored = int((~expected.eq(frame.patient_id)).sum())
        frame["patient_id"] = expected.astype("string")
        frame.attrs["patient_id_padding_restored_rows"] = restored
    return frame


def reject_final_partition(path: str | Path, frame: pd.DataFrame) -> None:
    """Prevent official-validation records from entering a training/selection stage."""
    if Path(path).name in {"valid.csv", "final_validation.csv"}:
        raise ValueError("Official validation cannot be used for training or development selection")
    paths = frame.image_path.astype(str).str.replace("\\", "/", regex=False)
    if paths.str.contains(r"(?:^|/)valid/", regex=True).any():
        raise ValueError(
            "Official validation images cannot be used for training or development selection"
        )


def select_patient_cohort(
    frame: pd.DataFrame, path: str | Path, fraction: float, seed: int
) -> pd.DataFrame:
    """Require and validate the persisted patient cohort shared by all methods."""
    cohort = json.loads(Path(path).read_text(encoding="utf-8"))
    if cohort.get("seed") != seed or cohort.get("label_fraction") != fraction:
        raise ValueError("Cohort seed/budget does not match the run configuration")
    identifiers = cohort.get("patient_ids", [])
    if not identifiers or any(not isinstance(value, str) for value in identifiers):
        raise ValueError("Cohort must contain nonempty string patient IDs")
    # Source paths supply the canonical spelling. Resolve legacy cohort IDs by
    # numeric identity without changing their membership, order, or file contents.
    available = {}
    for identifier in frame.patient_id.unique():
        identity = patient_identity(identifier)
        if identity in available and available[identity] != identifier:
            raise ValueError("Manifest contains ambiguous zero-padded patient IDs")
        available[identity] = identifier
    identities = [patient_identity(identifier) for identifier in identifiers]
    if len(set(identities)) != len(identifiers) or not set(identities).issubset(available):
        raise ValueError("Cohort contains duplicate or unknown patient IDs")
    patients = {available[identity] for identity in identities}
    rounding = cohort.get("budget_rounding", "round")
    if rounding not in {"round", "ceil"}:
        raise ValueError("Unknown cohort budget rounding")
    expected_count = max(
        1, (math.ceil if rounding == "ceil" else round)(frame.patient_id.nunique() * fraction)
    )
    if len(patients) != expected_count:
        raise ValueError("Cohort patient count does not match the configured budget")
    selected = frame.loc[frame.patient_id.isin(patients)].reset_index(drop=True)
    selected.attrs["cohort_patient_id_padding_restored"] = sum(
        identifier != available[identity]
        for identifier, identity in zip(identifiers, identities, strict=True)
    )
    return selected


def resolve_target(value: object, target: str) -> int:
    """Apply the declared CheXpert U-Ones/U-Zeros policy to one raw label."""
    if pd.isna(value):
        return 0
    value = int(value)
    if value == -1:
        return int(target in UNCERTAIN_AS_POSITIVE)
    if value not in (0, 1):
        raise ValueError(f"Unexpected label value {value!r} for {target}")
    return value


def local_image_path(raw_path: str, dataset_root: Path) -> Path:
    """Map a CheXpert CSV path to a portable path rooted at ``dataset_root``."""
    parts = Path(str(raw_path).replace("\\", "/")).parts
    root_name = dataset_root.name
    try:
        index = parts.index(root_name)
        relative = Path(*parts[index + 1 :])
    except ValueError:
        relative = Path(*parts[-3:]) if len(parts) >= 3 else Path(*parts)
    return dataset_root / relative


def extract_identifier(raw_path: str, pattern: re.Pattern[str], label: str) -> str:
    match = pattern.search(str(raw_path))
    if not match:
        raise ValueError(f"Could not extract {label} from {raw_path!r}")
    return match.group(1)


def prepare_manifest(
    csv_path: Path, dataset_root: Path, frontal_only: bool = False
) -> pd.DataFrame:
    """Load a CheXpert manifest and add portable paths, IDs, and resolved targets."""
    frame = pd.read_csv(csv_path)
    required = {"Path", "Frontal/Lateral", *TARGETS}
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f"Missing CheXpert columns: {sorted(missing)}")

    if frontal_only:
        frame = frame.loc[frame["Frontal/Lateral"].eq("Frontal")].copy()
    else:
        frame = frame.copy()

    frame["image_path"] = frame["Path"].map(lambda path: str(local_image_path(path, dataset_root)))
    frame["patient_id"] = frame["Path"].map(
        lambda path: extract_identifier(path, _PATIENT_RE, "patient ID")
    )
    frame["study_id"] = frame["Path"].map(
        lambda path: extract_identifier(path, _STUDY_RE, "study ID")
    )
    for target in TARGETS:
        frame[f"target_{target}"] = frame[target].map(
            lambda value, label=target: resolve_target(value, label)
        )
    return frame.reset_index(drop=True)


def assert_patient_disjoint(*frames: pd.DataFrame) -> None:
    """Raise when any two supplied manifests contain the same patient."""
    patient_sets = [set(frame["patient_id"].map(patient_identity)) for frame in frames]
    for index, current in enumerate(patient_sets):
        for other_index, other in enumerate(patient_sets[index + 1 :], start=index + 1):
            overlap = current.intersection(other)
            if overlap:
                raise ValueError(
                    f"Patient leakage between partitions {index} and {other_index}: "
                    f"{len(overlap)} shared patients"
                )


def exclude_development_patients(
    pretrain_manifest: pd.DataFrame, development_manifest: pd.DataFrame
) -> tuple[pd.DataFrame, int]:
    """Remove development patients from an existing pretraining manifest."""
    for label, frame in (("pretraining", pretrain_manifest), ("development", development_manifest)):
        if "patient_id" not in frame.columns:
            raise ValueError(f"{label} manifest must contain a patient_id column")
    development_ids = set(development_manifest["patient_id"].astype(str))
    mask = ~pretrain_manifest["patient_id"].astype(str).isin(development_ids)
    filtered = pretrain_manifest.loc[mask].copy().reset_index(drop=True)
    if filtered.empty:
        raise ValueError("No pretraining rows remain after excluding development patients")
    assert_patient_disjoint(filtered, development_manifest)
    removed = pretrain_manifest.loc[~mask, "patient_id"].astype(str).nunique()
    return filtered, int(removed)


def split_by_patient(
    frame: pd.DataFrame, dev_fraction: float, seed: int, random_state: bool = False
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Create a deterministic patient-disjoint development partition.

    Patients are assigned using a shuffled order, rather than individual studies,
    so the split remains safe even when a patient has multiple studies or views.
    Pre/post-split prevalence is recorded by ``prepare_data.py``.
    """
    if not 0 < dev_fraction < 1:
        raise ValueError("dev_fraction must be between 0 and 1")
    patients = np.array(sorted(frame["patient_id"].astype(str).unique()))
    if len(patients) < 2:
        raise ValueError("At least two patients are required for a train/development split")
    rng = np.random.RandomState(seed) if random_state else np.random.default_rng(seed)
    rng.shuffle(patients)
    dev_count = min(len(patients) - 1, max(1, round(len(patients) * dev_fraction)))
    dev_patients = set(patients[:dev_count])
    dev = frame.loc[frame["patient_id"].astype(str).isin(dev_patients)].copy()
    train = frame.loc[~frame["patient_id"].astype(str).isin(dev_patients)].copy()
    assert_patient_disjoint(train, dev)
    return train.reset_index(drop=True), dev.reset_index(drop=True)


def nested_patient_ids(
    frame: pd.DataFrame,
    fractions: Iterable[float],
    seed: int,
    random_state: bool = False,
    budget_rounding: str = "round",
) -> dict[float, list[str]]:
    """Return nested deterministic patient samples for every requested budget."""
    patients = np.array(sorted(frame["patient_id"].astype(str).unique()))
    if budget_rounding not in {"round", "ceil"}:
        raise ValueError("Unknown budget rounding")
    rng = np.random.RandomState(seed) if random_state else np.random.default_rng(seed)
    rng.shuffle(patients)
    result: dict[float, list[str]] = {}
    for fraction in sorted({float(value) for value in fractions}):
        if not 0 < fraction <= 1:
            raise ValueError("All fractions must be in (0, 1]")
        count = max(
            1, (math.ceil if budget_rounding == "ceil" else round)(len(patients) * fraction)
        )
        result[fraction] = patients[:count].tolist()
    return result


def manifest_hash(frame: pd.DataFrame) -> str:
    """Stable content hash used to tie artifacts back to their source manifest."""
    columns = [
        column
        for column in ("Path", "image_path", "patient_id", *target_columns())
        if column in frame
    ]
    payload = frame[columns].sort_values(columns).to_csv(index=False).encode("utf-8")
    return sha256(payload).hexdigest()


def target_columns() -> list[str]:
    return [f"target_{target}" for target in TARGETS]


class CheXpertDataset(Dataset[tuple[torch.Tensor, torch.Tensor]]):
    """Image dataset that returns an X-ray and its five resolved targets."""

    def __init__(self, frame: pd.DataFrame, transform: Callable | None = None) -> None:
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
        self.columns = target_columns()

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        row = self.frame.iloc[index]
        with Image.open(row["image_path"]) as source:
            image = source.convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        targets = torch.tensor(row[self.columns].to_numpy(dtype=np.float32), dtype=torch.float32)
        return image, targets


class CachedTargetDataset(Dataset):
    """Targets paired with bilinear-resized grayscale uint8 images from a memmap."""

    def __init__(
        self,
        frame: pd.DataFrame,
        cache_path: str,
        image_size: int,
        source_manifest: str | None = None,
    ):

        path = Path(cache_path)
        source = read_manifest(source_manifest) if source_manifest else frame
        reject_final_partition(source_manifest or path, source)
        lookup = {value: i for i, value in enumerate(source.image_path)}
        self.indices = [lookup[value] for value in frame.image_path]
        with artifact_lock(path.with_suffix(".lock")):
            self.images, _ = ensure_grayscale_cache(source, path, image_size)
        self.targets = torch.tensor(frame[target_columns()].to_numpy(dtype=np.float32))

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        return torch.from_numpy(
            np.array(self.images[self.indices[index]], copy=True)
        ), self.targets[index]


def supervised_transform(
    image_size: int, train: bool, config: dict | None = None
) -> transforms.Compose:
    config = config or {}
    if config.get("notebook_preprocessing"):
        normalization = config["normalization"]
        return transforms.Compose(
            [
                transforms.Grayscale(num_output_channels=3),
                transforms.Resize(
                    (image_size, image_size), interpolation=transforms.InterpolationMode.BILINEAR
                ),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=(normalization["mean"],) * 3, std=(normalization["std"],) * 3
                ),
            ]
        )
    operations: list[Callable] = [transforms.Resize((image_size, image_size))]
    if train:
        operations.append(
            transforms.RandomAffine(
                degrees=float(config.get("rotation", 5)),
                translate=tuple(config.get("translation", (0.02, 0.02))),
            )
        )
    operations.extend(
        [
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ]
    )
    return transforms.Compose(operations)


def existing_images(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return decodable images and a data-quality report for invalid records."""
    reasons: list[str] = []
    for value in frame["image_path"]:
        path = Path(value)
        if not path.is_file():
            reasons.append("missing")
            continue
        try:
            with Image.open(path) as image:
                image.verify()
            reasons.append("")
        except (OSError, ValueError):
            reasons.append("unreadable")
    report = frame.loc[[bool(reason) for reason in reasons]].copy()
    report["data_quality_reason"] = [reason for reason in reasons if reason]
    valid = frame.loc[[not bool(reason) for reason in reasons]].copy()
    return valid.reset_index(drop=True), report.reset_index(drop=True)


def prevalence(frame: pd.DataFrame) -> dict[str, float]:
    """Resolved target prevalence, including zero prevalence labels."""
    return {target: float(frame[f"target_{target}"].mean()) for target in TARGETS}

## Models

In [83]:
"""Encoders and downstream heads shared by SSL and supervised experiments."""


def build_encoder(
    name: str = "resnet18",
    pretrained: bool = False,
    weights_name: str | None = None,
    norm_layer=None,
) -> tuple[nn.Module, int]:
    """Return a supported torchvision backbone that emits one feature vector per image."""
    constructors = {
        "resnet18": (vision_models.resnet18, vision_models.ResNet18_Weights),
        "resnet50": (vision_models.resnet50, vision_models.ResNet50_Weights),
        "densenet121": (vision_models.densenet121, vision_models.DenseNet121_Weights),
        "efficientnet_b0": (vision_models.efficientnet_b0, vision_models.EfficientNet_B0_Weights),
        "vit_b_16": (vision_models.vit_b_16, vision_models.ViT_B_16_Weights),
    }
    if name not in constructors:
        supported = ", ".join(constructors)
        raise ValueError(f"Unsupported encoder {name!r}; choose one of: {supported}")

    constructor, weight_enum = constructors[name]
    weights = None
    if pretrained:
        weights = weight_enum[weights_name] if weights_name else weight_enum.DEFAULT
    model = constructor(weights=weights, **({"norm_layer": norm_layer} if norm_layer else {}))
    if name.startswith("resnet"):
        features = model.fc.in_features
        model.fc = nn.Identity()
    elif name == "densenet121":
        features = model.classifier.in_features
        model.classifier = nn.Identity()
    elif name == "efficientnet_b0":
        features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Identity()
    else:
        features = model.heads.head.in_features
        model.heads = nn.Identity()
    return model, features


class StandardizedHead(nn.Module):
    """Train-only feature statistics saved with the head for identical evaluation."""

    def __init__(self, feature_dim: int, num_labels: int):
        super().__init__()
        self.register_buffer("mean", torch.zeros(feature_dim))
        self.register_buffer("std", torch.ones(feature_dim))
        self.linear = nn.Linear(feature_dim, num_labels)

    def forward(self, features):
        return self.linear((features - self.mean) / self.std)


class MultiLabelClassifier(nn.Module):
    def __init__(
        self,
        encoder_name: str,
        num_labels: int,
        freeze_encoder: bool = False,
        pretrained: bool = False,
        weights_name: str | None = None,
        standardize_features: bool = False,
    ) -> None:
        super().__init__()
        self.encoder, feature_dim = build_encoder(
            encoder_name, pretrained=pretrained, weights_name=weights_name
        )
        self.classifier = (
            StandardizedHead(feature_dim, num_labels)
            if standardize_features
            else nn.Linear(feature_dim, num_labels)
        )
        if freeze_encoder:
            self.freeze_encoder()

    def freeze_encoder(self) -> None:
        for parameter in self.encoder.parameters():
            parameter.requires_grad = False

    def load_pretrained_encoder(self, checkpoint: dict[str, object]) -> None:
        """Load the online encoder from any SSL method checkpoint."""
        state = checkpoint.get("encoder")
        if state is None:
            model_state = checkpoint.get("model", checkpoint)
            state = {
                key.removeprefix("encoder."): value
                for key, value in model_state.items()
                if key.startswith("encoder.")
            }
        if not state:
            raise ValueError("Checkpoint does not contain transferable online encoder weights")
        encoder_state = state
        self.encoder.load_state_dict(encoder_state, strict=True)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.encoder(images))

## Gpu Augment

In [84]:
"""Batched X-ray augmentations applied on the training device."""


def _uniform(batch: int, bounds: tuple[float, float], device: torch.device) -> torch.Tensor:
    return torch.rand(batch, device=device) * (bounds[1] - bounds[0]) + bounds[0]


@torch.no_grad()
def augment_grayscale_batch(
    images: torch.Tensor,
    image_size: int,
    config: dict,
    channels_last: bool = True,
) -> torch.Tensor:
    """Augment uint8 images shaped (B,H,W), returning normalized RGB tensors."""
    if images.ndim != 3 or images.dtype != torch.uint8:
        raise ValueError("Expected a uint8 image batch shaped (batch, height, width)")
    device = images.device
    batch = images.shape[0]
    x = images.to(dtype=torch.float32).unsqueeze(1).div_(255.0)

    area = _uniform(batch, tuple(config.get("scale", (0.6, 1.0))), device)
    aspect = torch.exp(
        _uniform(
            batch,
            tuple(math.log(value) for value in config.get("ratio", (0.85, 1.18))),
            device,
        )
    )
    crop_w = torch.sqrt(area * aspect).clamp(max=1.0)
    crop_h = torch.sqrt(area / aspect).clamp(max=1.0)
    angle = _uniform(
        batch, (-float(config.get("rotation", 10.0)), float(config.get("rotation", 10.0))), device
    ) * (math.pi / 180.0)
    translation = float(config.get("translation", 0.05))
    offset_x = (torch.rand(batch, device=device) * 2 - 1) * (1 - crop_w)
    offset_x += (torch.rand(batch, device=device) * 2 - 1) * translation * 2
    offset_y = (torch.rand(batch, device=device) * 2 - 1) * (1 - crop_h)
    offset_y += (torch.rand(batch, device=device) * 2 - 1) * translation * 2
    cosine, sine = torch.cos(angle), torch.sin(angle)
    flip = torch.ones(batch, device=device)
    if bool(config.get("horizontal_flip", False)):
        flip = torch.where(torch.rand(batch, device=device) < 0.5, -flip, flip)
    theta = torch.stack(
        (
            torch.stack((cosine * crop_w * flip, -sine * crop_h, offset_x), dim=1),
            torch.stack((sine * crop_w * flip, cosine * crop_h, offset_y), dim=1),
        ),
        dim=1,
    )
    grid = F.affine_grid(theta, (batch, 1, image_size, image_size), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="border", align_corners=False)

    contrast_range = float(config.get("contrast", 0.2))
    brightness_range = float(config.get("brightness", 0.1))
    contrast = 1 + _uniform(batch, (-contrast_range, contrast_range), device).view(batch, 1, 1, 1)
    brightness = _uniform(batch, (-brightness_range, brightness_range), device).view(batch, 1, 1, 1)
    mean = x.mean(dim=(1, 2, 3), keepdim=True)
    x = ((x - mean) * contrast + mean + brightness).clamp_(0, 1)

    blur_probability = float(config.get("blur_probability", 0.5))
    if blur_probability > 0:
        kernel_size = 9
        radius = kernel_size // 2
        sigma_bounds = tuple(config.get("blur_sigma", (0.1, 1.5)))
        blur_mask = torch.rand(batch, device=device) < blur_probability
        sigma = torch.where(
            blur_mask,
            _uniform(batch, sigma_bounds, device),
            torch.full((batch,), 1e-3, device=device),
        )
        positions = torch.arange(-radius, radius + 1, device=device, dtype=torch.float32)
        kernel = torch.exp(-(positions[None, :] ** 2) / (2 * sigma[:, None] ** 2))
        kernel /= kernel.sum(dim=1, keepdim=True)
        x = x.reshape(1, batch, image_size, image_size)
        horizontal = kernel.view(batch, 1, 1, kernel_size)
        vertical = kernel.view(batch, 1, kernel_size, 1)
        x = F.conv2d(F.pad(x, (radius, radius, 0, 0), mode="reflect"), horizontal, groups=batch)
        x = F.conv2d(F.pad(x, (0, 0, radius, radius), mode="reflect"), vertical, groups=batch)
        x = x.reshape(batch, 1, image_size, image_size)

    noise_probability = float(config.get("noise_probability", 0.5))
    noise_std = float(config.get("noise_std", 0.02))
    if noise_probability > 0 and noise_std > 0:
        mask = (torch.rand(batch, device=device) < noise_probability).view(batch, 1, 1, 1)
        std = torch.rand(batch, device=device).view(batch, 1, 1, 1) * noise_std * mask
        x = (x + torch.randn_like(x) * std).clamp_(0, 1)

    normalization = config.get("normalization", {"mean": 0.5, "std": 0.5})
    x = x.sub(float(normalization["mean"])).div(float(normalization["std"]))
    x = x.expand(-1, 3, -1, -1)
    return x.contiguous(
        memory_format=torch.channels_last if channels_last else torch.contiguous_format
    )

## Sampling

In [85]:
"""Epoch-reconstructible batch orders used by the reference notebook."""


class NotebookBatchSampler(Sampler):
    def __init__(self, records: int, batch_size: int, seed: int, protocol: str):
        self.records, self.batch_size, self.seed = records, batch_size, seed
        self.protocol, self.epoch = protocol, 0
        if protocol not in {"ssl", "finetune", "linear"}:
            raise ValueError("Unknown notebook sampling protocol")

    def __len__(self):
        return (
            self.records // self.batch_size
            if self.protocol == "ssl"
            else math.ceil(self.records / self.batch_size)
        )

    def __iter__(self):
        if self.protocol == "ssl":
            order = np.random.RandomState(self.seed * 1000 + self.epoch).permutation(self.records)
        elif self.protocol == "finetune":
            rng = np.random.RandomState(self.seed)
            for _ in range(self.epoch + 1):
                order = rng.permutation(self.records)
        else:
            generator = torch.Generator().manual_seed(self.seed)
            for _ in range(self.epoch + 1):
                order = torch.randperm(self.records, generator=generator).numpy()
        for i in range(len(self)):
            batch = order[i * self.batch_size : (i + 1) * self.batch_size]
            yield (batch if self.protocol == "linear" else np.sort(batch)).tolist()

## Image Cache

In [86]:
"""Disk-backed grayscale image cache for repeated SSL training epochs."""


CACHE_VERSION = 1


class CachedGrayscaleDataset(Dataset):
    """Read resized uint8 grayscale images from a memory-mapped NumPy cache."""

    def __init__(self, cache_path: Path, image_count: int) -> None:
        self.cache_path = Path(cache_path)
        self.image_count = image_count
        self._images: np.ndarray | None = None

    def __len__(self) -> int:
        return self.image_count

    def __getstate__(self) -> dict:
        # Spawned workers reopen the mmap instead of serializing the entire cache.
        return {**self.__dict__, "_images": None}

    def __getitem__(self, index: int) -> np.ndarray:
        if self._images is None:
            self._images = np.load(self.cache_path, mmap_mode="r")
        return np.array(self._images[index], dtype=np.uint8, copy=True)


def _decode_image(index: int, image_path: str, resolution: int) -> tuple[int, np.ndarray]:
    with Image.open(image_path) as image:
        resized = image.convert("L").resize(
            (resolution, resolution), resample=Image.Resampling.BILINEAR
        )
        pixels = np.asarray(resized, dtype=np.uint8)
    return index, pixels


def ensure_grayscale_cache(
    frame: pd.DataFrame,
    cache_path: Path,
    resolution: int,
    workers: int = 16,
    log_interval: int = 10_000,
) -> tuple[CachedGrayscaleDataset, bool]:
    """Build or reuse an image cache keyed by manifest contents and resolution."""
    if "image_path" not in frame:
        raise ValueError("Pretraining manifest must contain an image_path column")
    if resolution < 32:
        raise ValueError("Cache resolution must be at least 32 pixels")
    if workers < 1:
        raise ValueError("Cache workers must be at least 1")

    cache_path = Path(cache_path)
    metadata_path = cache_path.with_suffix(".json")
    signature = {
        "cache_version": CACHE_VERSION,
        "resolution": resolution,
        "records": len(frame),
        "manifest_hash": manifest_hash(frame),
    }
    if cache_path.is_file() and metadata_path.is_file():
        try:
            metadata: dict[str, Any] = json.loads(metadata_path.read_text(encoding="utf-8"))
            cached = np.load(cache_path, mmap_mode="r")
            if (
                metadata == signature
                and cached.shape == (len(frame), resolution, resolution)
                and cached.dtype == np.uint8
            ):
                print(f"Reusing grayscale image cache: {cache_path}", flush=True)
                return CachedGrayscaleDataset(cache_path, len(frame)), False
        except (OSError, ValueError, json.JSONDecodeError):
            pass

    cache_path.parent.mkdir(parents=True, exist_ok=True)
    required_bytes = len(frame) * resolution * resolution + 1_048_576
    free_bytes = shutil.disk_usage(cache_path.parent).free
    if free_bytes < required_bytes:
        required_gib = required_bytes / 2**30
        free_gib = free_bytes / 2**30
        raise OSError(
            f"Image cache needs about {required_gib:.2f} GiB but only "
            f"{free_gib:.2f} GiB is free at {cache_path.parent}"
        )
    temporary_path = cache_path.with_name(f"{cache_path.stem}.building.npy")
    images: np.memmap | None = np.lib.format.open_memmap(
        temporary_path,
        mode="w+",
        dtype=np.uint8,
        shape=(len(frame), resolution, resolution),
    )
    jobs = iter((index, str(path), resolution) for index, path in enumerate(frame["image_path"]))
    completed = 0
    try:
        with ThreadPoolExecutor(max_workers=min(workers, max(1, len(frame)))) as executor:
            while batch := list(islice(jobs, 2048)):
                for index, pixels in executor.map(lambda job: _decode_image(*job), batch):
                    images[index] = pixels
                    completed += 1
                    if completed % log_interval == 0 or completed == len(frame):
                        print(f"cache images={completed}/{len(frame)}", flush=True)
        images.flush()
        del images
        images = None
        temporary_path.replace(cache_path)
        metadata_temporary = metadata_path.with_suffix(".json.tmp")
        metadata_temporary.write_text(json.dumps(signature, indent=2), encoding="utf-8")
        metadata_temporary.replace(metadata_path)
    except Exception:
        if images is not None:
            del images
        temporary_path.unlink(missing_ok=True)
        raise

    print(f"Built grayscale image cache: {cache_path}", flush=True)
    return CachedGrayscaleDataset(cache_path, len(frame)), True

## Ssl Methods

In [87]:
"""Five image-only self-supervised objectives sharing the project encoder."""


METHODS = ("simclr", "moco", "byol", "nnclr", "swav")


def nt_xent_loss(
    view_one: torch.Tensor, view_two: torch.Tensor, temperature: float = 0.2
) -> torch.Tensor:
    """Normalized temperature-scaled cross-entropy loss for paired SimCLR views."""
    if view_one.shape != view_two.shape:
        raise ValueError("SimCLR views must have identical embedding shapes")
    if view_one.shape[0] < 2:
        raise ValueError("NT-Xent requires at least two samples per batch")
    if temperature <= 0:
        raise ValueError("Temperature must be positive")
    embeddings = F.normalize(torch.cat([view_one, view_two], dim=0), dim=1)
    logits = embeddings @ embeddings.T / temperature
    logits.fill_diagonal_(float("-inf"))
    batch_size = view_one.shape[0]
    positives = torch.cat(
        [torch.arange(batch_size, 2 * batch_size), torch.arange(0, batch_size)]
    ).to(embeddings.device)
    return F.cross_entropy(logits, positives)


class SplitBatchNorm(nn.BatchNorm2d):
    """Notebook's single-device MoCo normalization over independent subgroups."""

    def __init__(self, num_features: int, num_splits: int = 4, **kwargs):
        super().__init__(num_features, **kwargs)
        if num_splits < 1:
            raise ValueError("BatchNorm splits must be positive")
        self.num_splits = num_splits

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        n, c, h, w = inputs.shape
        if self.training and n % self.num_splits == 0:
            mean = self.running_mean.repeat(self.num_splits)
            var = self.running_var.repeat(self.num_splits)
            output = F.batch_norm(
                inputs.reshape(-1, c * self.num_splits, h, w),
                mean,
                var,
                self.weight.repeat(self.num_splits),
                self.bias.repeat(self.num_splits),
                True,
                self.momentum,
                self.eps,
            ).reshape(n, c, h, w)
            with torch.no_grad():
                self.running_mean.copy_(mean.reshape(self.num_splits, c).mean(0))
                self.running_var.copy_(var.reshape(self.num_splits, c).mean(0))
            return output
        return F.batch_norm(
            inputs,
            self.running_mean,
            self.running_var,
            self.weight,
            self.bias,
            False,
            self.momentum,
            self.eps,
        )


def _mlp(input_dim: int, hidden_dim: int, output_dim: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Linear(input_dim, hidden_dim, bias=False),
        nn.BatchNorm1d(hidden_dim),
        nn.ReLU(inplace=True),
        nn.Linear(hidden_dim, output_dim),
    )


def _normalize(output: torch.Tensor) -> torch.Tensor:
    return F.normalize(output.float(), dim=1)


def _info_nce(prediction: torch.Tensor, positive: torch.Tensor, temperature: float) -> torch.Tensor:
    return F.cross_entropy(
        prediction @ positive.T / temperature,
        torch.arange(prediction.shape[0], device=prediction.device),
    )


@torch.no_grad()
def _ema_update(target: nn.Module, online: nn.Module, momentum: float) -> None:
    for target_parameter, online_parameter in zip(target.parameters(), online.parameters()):
        target_parameter.lerp_(online_parameter.detach(), 1.0 - momentum)


class SSLMethod(nn.Module):
    def after_optimizer_step(self, progress: float) -> None:
        del progress


class SimCLR(SSLMethod):
    def __init__(self, encoder_name: str, projection_dim: int, hidden_dim: int, temperature: float):
        super().__init__()
        self.encoder, feature_dim = build_encoder(encoder_name)
        self.projector = _mlp(feature_dim, hidden_dim, projection_dim)
        self.temperature = temperature

    def forward(self, first: torch.Tensor, second: torch.Tensor) -> torch.Tensor:
        batch = first.shape[0]
        projections = self.projector(self.encoder(torch.cat((first, second)))).float()
        return nt_xent_loss(projections[:batch], projections[batch:], self.temperature)


class MoCo(SSLMethod):
    def __init__(
        self,
        encoder_name: str,
        projection_dim: int,
        hidden_dim: int,
        temperature: float,
        queue_size: int,
        momentum: float,
        bn_splits: int = 1,
    ):
        super().__init__()
        self.encoder, feature_dim = build_encoder(
            encoder_name,
            norm_layer=(lambda c: SplitBatchNorm(c, bn_splits)) if bn_splits > 1 else None,
        )
        self.projector = _mlp(feature_dim, hidden_dim, projection_dim)
        self.key_encoder = copy.deepcopy(self.encoder)
        self.key_projector = copy.deepcopy(self.projector)
        for parameter in (*self.key_encoder.parameters(), *self.key_projector.parameters()):
            parameter.requires_grad_(False)
        self.temperature, self.momentum, self.queue_size = temperature, momentum, queue_size
        self.register_buffer("queue", F.normalize(torch.randn(queue_size, projection_dim), dim=1))
        self.register_buffer("queue_pointer", torch.zeros(1, dtype=torch.long))

    @torch.no_grad()
    def _keys(self, images: torch.Tensor) -> torch.Tensor:
        permutation = torch.randperm(images.shape[0], device=images.device)
        keys = _normalize(self.key_projector(self.key_encoder(images[permutation])))
        return keys[torch.argsort(permutation)]

    @torch.no_grad()
    def _enqueue(self, keys: torch.Tensor) -> None:
        count, pointer = keys.shape[0], int(self.queue_pointer)
        positions = (pointer + torch.arange(count, device=keys.device)) % self.queue_size
        self.queue[positions] = keys
        self.queue_pointer[0] = (pointer + count) % self.queue_size

    def _loss(self, query: torch.Tensor, key: torch.Tensor, queue: torch.Tensor) -> torch.Tensor:
        positive = (query * key).sum(dim=1, keepdim=True)
        logits = torch.cat((positive, query @ queue.T), dim=1) / self.temperature
        return F.cross_entropy(
            logits, torch.zeros(query.shape[0], dtype=torch.long, device=query.device)
        )

    def forward(self, first: torch.Tensor, second: torch.Tensor) -> torch.Tensor:
        queue = self.queue.detach().clone()
        q1 = _normalize(self.projector(self.encoder(first)))
        q2 = _normalize(self.projector(self.encoder(second)))
        k1, k2 = self._keys(first), self._keys(second)
        loss = 0.5 * (self._loss(q1, k2, queue) + self._loss(q2, k1, queue))
        self._enqueue(torch.cat((k1, k2)))
        return loss

    def after_optimizer_step(self, progress: float) -> None:
        del progress
        _ema_update(self.key_encoder, self.encoder, self.momentum)
        _ema_update(self.key_projector, self.projector, self.momentum)


class BYOL(SSLMethod):
    def __init__(self, encoder_name: str, projection_dim: int, hidden_dim: int, momentum: float):
        super().__init__()
        self.encoder, feature_dim = build_encoder(encoder_name)
        self.projector = _mlp(feature_dim, hidden_dim, projection_dim)
        self.predictor = _mlp(projection_dim, hidden_dim, projection_dim)
        self.target_encoder = copy.deepcopy(self.encoder)
        self.target_projector = copy.deepcopy(self.projector)
        for parameter in (*self.target_encoder.parameters(), *self.target_projector.parameters()):
            parameter.requires_grad_(False)
        self.base_momentum = momentum

    def forward(self, first: torch.Tensor, second: torch.Tensor) -> torch.Tensor:
        p1 = self.predictor(self.projector(self.encoder(first)))
        p2 = self.predictor(self.projector(self.encoder(second)))
        with torch.no_grad():
            t1 = self.target_projector(self.target_encoder(first))
            t2 = self.target_projector(self.target_encoder(second))
        loss1 = 2 - 2 * (_normalize(p1) * _normalize(t2)).sum(dim=1).mean()
        loss2 = 2 - 2 * (_normalize(p2) * _normalize(t1)).sum(dim=1).mean()
        return loss1 + loss2

    def after_optimizer_step(self, progress: float) -> None:
        momentum = 1 - (1 - self.base_momentum) * (math.cos(math.pi * progress) + 1) / 2
        _ema_update(self.target_encoder, self.encoder, momentum)
        _ema_update(self.target_projector, self.projector, momentum)


class NNCLR(SSLMethod):
    def __init__(
        self,
        encoder_name: str,
        projection_dim: int,
        hidden_dim: int,
        temperature: float,
        queue_size: int,
        positive_as_anchor: bool = False,
    ):
        super().__init__()
        self.encoder, feature_dim = build_encoder(encoder_name)
        self.projector = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim, bias=False),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim, bias=False),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, projection_dim),
        )
        self.predictor = _mlp(projection_dim, hidden_dim, projection_dim)
        self.temperature, self.queue_size = temperature, queue_size
        self.positive_as_anchor = positive_as_anchor
        self.register_buffer("support", F.normalize(torch.randn(queue_size, projection_dim), dim=1))
        self.register_buffer("support_filled", torch.zeros(1, dtype=torch.long))
        self.register_buffer("support_pointer", torch.zeros(1, dtype=torch.long))

    @torch.no_grad()
    def _nearest(self, embeddings: torch.Tensor) -> torch.Tensor:
        if int(self.support_filled) < self.queue_size:
            return embeddings
        return self.support[(embeddings @ self.support.T).argmax(dim=1)]

    @torch.no_grad()
    def _enqueue(self, embeddings: torch.Tensor) -> None:
        count, pointer = embeddings.shape[0], int(self.support_pointer)
        positions = (pointer + torch.arange(count, device=embeddings.device)) % self.queue_size
        self.support[positions] = embeddings
        self.support_pointer[0] = (pointer + count) % self.queue_size
        self.support_filled[0] = min(self.queue_size, int(self.support_filled) + count)

    def forward(self, first: torch.Tensor, second: torch.Tensor) -> torch.Tensor:
        h1, h2 = self.projector(self.encoder(first)), self.projector(self.encoder(second))
        z1, z2 = _normalize(h1.detach()), _normalize(h2.detach())
        p1, p2 = _normalize(self.predictor(h1)), _normalize(self.predictor(h2))
        positive1, positive2 = self._nearest(z1), self._nearest(z2)
        loss = (
            0.5
            * (
                _info_nce(positive1, p2, self.temperature)
                + _info_nce(positive2, p1, self.temperature)
            )
            if self.positive_as_anchor
            else 0.5
            * (
                _info_nce(p2, positive1, self.temperature)
                + _info_nce(p1, positive2, self.temperature)
            )
        )
        self._enqueue(torch.cat((z1, z2)))
        return loss


@torch.no_grad()
def _sinkhorn(scores: torch.Tensor, epsilon: float, iterations: int = 3) -> torch.Tensor:
    if epsilon <= 0 or iterations < 1:
        raise ValueError("Sinkhorn epsilon and iterations must be positive")
    # Log-space float32 normalization prevents exp overflow under mixed precision.
    assignments = scores.float().T / epsilon
    assignments -= torch.logsumexp(assignments.flatten(), dim=0)
    prototypes, batch = assignments.shape
    for _ in range(iterations):
        assignments -= torch.logsumexp(assignments, dim=1, keepdim=True)
        assignments -= math.log(prototypes)
        assignments -= torch.logsumexp(assignments, dim=0, keepdim=True)
        assignments -= math.log(batch)
    return torch.exp(assignments + math.log(batch)).T


class SwAV(SSLMethod):
    def __init__(
        self,
        encoder_name: str,
        projection_dim: int,
        hidden_dim: int,
        temperature: float,
        prototype_count: int,
        epsilon: float,
        freeze_steps: int,
    ):
        super().__init__()
        self.encoder, feature_dim = build_encoder(encoder_name)
        self.projector = _mlp(feature_dim, hidden_dim, projection_dim)
        self.prototypes = nn.Parameter(
            F.normalize(torch.randn(prototype_count, projection_dim), dim=1)
        )
        self.temperature, self.epsilon, self.freeze_prototype_steps = (
            temperature,
            epsilon,
            freeze_steps,
        )
        self.register_buffer("step", torch.zeros(1, dtype=torch.long))

    def forward(self, first: torch.Tensor, second: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            self.prototypes.copy_(F.normalize(self.prototypes, dim=1))
        weight = (
            self.prototypes.detach()
            if int(self.step) < self.freeze_prototype_steps
            else self.prototypes
        )
        batch = first.shape[0]
        embeddings = _normalize(self.projector(self.encoder(torch.cat((first, second)))))
        scores = embeddings @ weight.float().T
        first_scores, second_scores = scores[:batch], scores[batch:]
        first_assign = _sinkhorn(first_scores.detach(), self.epsilon)
        second_assign = _sinkhorn(second_scores.detach(), self.epsilon)
        loss1 = (
            -(first_assign * F.log_softmax(second_scores / self.temperature, dim=1)).sum(1).mean()
        )
        loss2 = (
            -(second_assign * F.log_softmax(first_scores / self.temperature, dim=1)).sum(1).mean()
        )
        return 0.5 * (loss1 + loss2)

    def after_optimizer_step(self, progress: float) -> None:
        del progress
        self.step += 1


def build_ssl_method(name: str, encoder_name: str, config: dict) -> SSLMethod:
    """Build one method from the common YAML SSL settings."""
    if name not in METHODS:
        raise ValueError(f"Unknown SSL method {name!r}; choose from {', '.join(METHODS)}")
    common = (
        encoder_name,
        int(config.get("projection_dim", 128)),
        int(config.get("hidden_dim", 1024)),
    )
    temperature = float(config.get("temperatures", {}).get(name, config.get("temperature", 0.2)))
    if name == "simclr":
        return SimCLR(*common, temperature)
    if name == "moco":
        return MoCo(
            *common,
            temperature,
            int(config.get("queue_size", 4096)),
            float(config.get("momentum", 0.99)),
            int(config.get("bn_splits", 1)),
        )
    if name == "byol":
        return BYOL(*common, float(config.get("momentum", 0.99)))
    if name == "nnclr":
        return NNCLR(
            *common,
            temperature,
            int(config.get("queue_size", 8192)),
            bool(config.get("positive_as_anchor", False)),
        )
    return SwAV(
        *common,
        temperature,
        int(config.get("prototype_count", 300)),
        float(config.get("sinkhorn_epsilon", 0.05)),
        int(config.get("prototype_freeze_steps", 100)),
    )


def encoder_state_dict(method: SSLMethod) -> dict[str, torch.Tensor]:
    """Return the online encoder weights saved for downstream transfer."""
    return method.encoder.state_dict()

## Metrics

In [88]:
"""Multi-label metrics with explicit handling for single-class labels."""


def multilabel_metrics(targets: np.ndarray, probabilities: np.ndarray) -> dict[str, float]:
    if targets.shape != probabilities.shape:
        raise ValueError("targets and probabilities must have the same shape")
    if targets.shape[1] != len(TARGETS):
        raise ValueError(f"Expected {len(TARGETS)} label columns")

    results: dict[str, float] = {}
    aurocs: list[float] = []
    auprcs: list[float] = []
    for index, label in enumerate(TARGETS):
        truth, scores = targets[:, index], probabilities[:, index]
        if len(np.unique(truth)) < 2:
            results[f"auroc_{label}"] = float("nan")
            results[f"auprc_{label}"] = float("nan")
            continue
        auroc = float(roc_auc_score(truth, scores))
        auprc = float(average_precision_score(truth, scores))
        results[f"auroc_{label}"] = auroc
        results[f"auprc_{label}"] = auprc
        aurocs.append(auroc)
        auprcs.append(auprc)
    results["macro_auroc"] = float(np.mean(aurocs)) if aurocs else float("nan")
    results["macro_auprc"] = float(np.mean(auprcs)) if auprcs else float("nan")
    return results


def select_thresholds(
    targets: np.ndarray, probabilities: np.ndarray, method: str = "balanced_accuracy"
) -> dict[str, float]:
    """Choose per-label balanced-accuracy thresholds on development data only."""
    thresholds: dict[str, float] = {}
    for index, label in enumerate(TARGETS):
        truth, scores = targets[:, index], probabilities[:, index]
        if len(np.unique(truth)) < 2:
            thresholds[label] = 0.5
            continue
        if method == "youden":
            fpr, tpr, values = roc_curve(truth, scores)
            value = values[int(np.argmax(tpr - fpr))]
            thresholds[label] = float(value) if np.isfinite(value) else 0.5
            continue
        if method != "balanced_accuracy":
            raise ValueError("Unknown threshold selection method")
        candidates = np.unique(np.concatenate(([0.0, 0.5, 1.0], scores)))
        values = [balanced_accuracy_score(truth, scores >= value) for value in candidates]
        thresholds[label] = float(candidates[int(np.argmax(values))])
    return thresholds


def threshold_metrics(
    targets: np.ndarray,
    probabilities: np.ndarray,
    thresholds: dict[str, float],
    notebook: bool = False,
) -> dict[str, float]:
    """Return thresholded metrics with safe NaNs for single-class labels."""
    result: dict[str, float] = {}
    f1s: list[float] = []
    balanced: list[float] = []
    for index, label in enumerate(TARGETS):
        truth = targets[:, index]
        predicted = probabilities[:, index] >= thresholds[label]
        if notebook:
            positive = truth == 1
            negative = ~positive
            tp, fn = int((predicted & positive).sum()), int((~predicted & positive).sum())
            tn, fp = int((~predicted & negative).sum()), int((predicted & negative).sum())
            sensitivity = tp / (tp + fn) if tp + fn else float("nan")
            specificity = tn / (tn + fp) if tn + fp else float("nan")
            f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else float("nan")
            rates = [v for v in (sensitivity, specificity) if np.isfinite(v)]
            balanced_score = float(np.mean(rates)) if rates else float("nan")
            result.update(
                {
                    f"sensitivity_{label}": sensitivity,
                    f"specificity_{label}": specificity,
                    f"f1_{label}": f1,
                    f"balanced_accuracy_{label}": balanced_score,
                }
            )
            if np.isfinite(f1):
                f1s.append(f1)
            if np.isfinite(balanced_score):
                balanced.append(balanced_score)
            continue
        if len(np.unique(truth)) < 2:
            for metric in ("sensitivity", "specificity", "f1", "balanced_accuracy"):
                result[f"{metric}_{label}"] = float("nan")
            continue
        positives = truth == 1
        negatives = ~positives
        result[f"sensitivity_{label}"] = float(predicted[positives].mean())
        result[f"specificity_{label}"] = float((~predicted[negatives]).mean())
        result[f"f1_{label}"] = float(f1_score(truth, predicted, zero_division=0))
        result[f"balanced_accuracy_{label}"] = float(balanced_accuracy_score(truth, predicted))
        f1s.append(result[f"f1_{label}"])
        balanced.append(result[f"balanced_accuracy_{label}"])
    result["macro_f1"] = float(np.mean(f1s)) if f1s else float("nan")
    result["macro_balanced_accuracy"] = float(np.mean(balanced)) if balanced else float("nan")
    return result


def patient_bootstrap_macro_auroc(
    targets: np.ndarray,
    probabilities: np.ndarray,
    patient_ids: np.ndarray,
    seed: int,
    samples: int = 1000,
) -> tuple[float, float]:
    """Patient-resampled percentile CI for macro AUROC."""
    patient_ids = np.asarray(patient_ids).astype(str)
    unique = np.unique(patient_ids)
    rng = np.random.default_rng(seed)
    values: list[float] = []
    for _ in range(samples):
        selected = rng.choice(unique, size=len(unique), replace=True)
        indices = np.concatenate([np.flatnonzero(patient_ids == patient) for patient in selected])
        score = multilabel_metrics(targets[indices], probabilities[indices])["macro_auroc"]
        if np.isfinite(score):
            values.append(score)
    if not values:
        return float("nan"), float("nan")
    return tuple(float(value) for value in np.percentile(values, [2.5, 97.5]))


def patient_bootstrap_auroc(targets, probabilities, patient_ids, seed, samples=1000):
    """Notebook-style macro and pathology CIs from the same patient resamples."""
    patient_ids = np.asarray(patient_ids).astype(str)
    groups = [np.flatnonzero(patient_ids == p) for p in np.unique(patient_ids)]
    rng = np.random.RandomState(seed)
    values = {key: [] for key in ["macro_auroc", *(f"auroc_{l}" for l in TARGETS)]}
    for _ in range(samples):
        indices = np.concatenate([groups[k] for k in rng.randint(0, len(groups), len(groups))])
        metrics = multilabel_metrics(targets[indices], probabilities[indices])
        for key, collected in values.items():
            if np.isfinite(metrics[key]):
                collected.append(metrics[key])
    intervals = {
        key: tuple(float(v) for v in np.percentile(scores, [2.5, 97.5]))
        if scores
        else (float("nan"), float("nan"))
        for key, scores in values.items()
    }
    return {
        "macro_auroc": intervals["macro_auroc"],
        "per_label": {label: intervals[f"auroc_{label}"] for label in TARGETS},
    }

## Utils

Atomic checkpoints, run locks, resolved configuration, software provenance and strict resume signatures.

In [89]:
"""Configuration, reproducibility, and lightweight checkpoint utilities."""


@contextmanager
def artifact_lock(path: Path):
    """Hold an OS-backed lock; crashes release ownership without stale-lock cleanup."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a+b") as handle:
        if os.name == "nt":
            import msvcrt

            handle.seek(0)
            if not handle.read(1):
                handle.write(b"0")
                handle.flush()
            handle.seek(0)
            try:
                msvcrt.locking(handle.fileno(), msvcrt.LK_NBLCK, 1)
            except OSError as error:
                raise RuntimeError(f"Another process owns this artifact: {path}") from error
            try:
                yield
            finally:
                handle.seek(0)
                msvcrt.locking(handle.fileno(), msvcrt.LK_UNLCK, 1)
        else:
            import fcntl

            try:
                fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)
            except BlockingIOError as error:
                raise RuntimeError(f"Another process owns this artifact: {path}") from error
            try:
                yield
            finally:
                fcntl.flock(handle, fcntl.LOCK_UN)


def managed_run(function):
    """Reject concurrent writers and record failures after a run has started."""

    @wraps(function)
    def wrapped(config: dict):
        with project_directory():
            config = {
                **config,
                "implementation_sha256": implementation_hash(Path(function.__code__.co_filename)),
            }
            output_dir = Path(config["output_dir"])
            with artifact_lock(output_dir / ".run.lock"):
                metadata_path = output_dir / "run_metadata.json"
                previous = metadata_path.read_bytes() if metadata_path.exists() else None
                try:
                    return function(config)
                except (Exception, KeyboardInterrupt) as error:
                    if metadata_path.exists() and metadata_path.read_bytes() != previous:
                        metadata = load_config(metadata_path)
                        if metadata.get("status") == "running":
                            metadata.update(
                                status="interrupted"
                                if isinstance(error, KeyboardInterrupt)
                                else "failed",
                                error=f"{type(error).__name__}: {error}",
                                ended_at_utc=datetime.now(timezone.utc).isoformat(),
                            )
                            save_json(metadata_path, metadata)
                    raise

    return wrapped


def merge_config(base: dict, overrides: dict) -> dict:
    result = dict(base)
    for key, value in overrides.items():
        result[key] = (
            merge_config(result[key], value)
            if isinstance(value, dict) and isinstance(result.get(key), dict)
            else value
        )
    return result


def normalize_config(config: dict) -> dict:
    """Resolve numeric experiment settings consistently across YAML and JSON.

    YAML loaders can interpret scientific notation such as ``1e-05`` as a
    string. Normalize declared numeric keys rather than arbitrary text values.
    """
    float_keys = {
        "learning_rate",
        "encoder_learning_rate",
        "weight_decay",
        "label_fraction",
        "temperature",
        "momentum",
        "sinkhorn_epsilon",
        "dev_fraction",
        "mean",
        "std",
        "pos_weight_max",
    }
    integer_keys = {
        "seed",
        "batch_size",
        "epochs",
        "image_size",
        "num_workers",
        "prefetch_factor",
        "cache_workers",
        "cache_log_interval",
        "torch_num_threads",
        "early_stopping_patience",
        "bootstrap_samples",
        "hidden_dim",
        "projection_dim",
        "queue_size",
        "prototype_count",
        "prototype_freeze_steps",
        "prototype_freeze_epochs",
        "warmup_epochs",
        "bn_splits",
        "max_steps_per_epoch",
    }
    resolved = {}
    for key, value in config.items():
        if isinstance(value, dict):
            value = normalize_config(value)
        elif isinstance(value, list):
            value = [normalize_config(item) if isinstance(item, dict) else item for item in value]
        if value is not None and key in float_keys:
            value = float(value)
        elif value is not None and key in integer_keys:
            numeric = float(value)
            if not math.isfinite(numeric) or not numeric.is_integer():
                raise ValueError(f"{key} must be an integer")
            value = int(numeric)
        resolved[key] = value
    return resolved


@lru_cache(maxsize=2)
def embedded_configs(source):
    """Cache parsing of the YAML bundle while respecting configuration-cell edits."""
    return yaml.safe_load(source)


def load_config(path: Path, _seen: tuple[Path, ...] = ()) -> dict[str, Any]:
    path = Path(path)
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    path = path.resolve()
    if path in _seen:
        raise ValueError(f"Circular configuration inheritance: {path}")
    try:
        key = path.relative_to(PROJECT_ROOT).as_posix()
    except ValueError:
        key = None
    bundle = embedded_configs(CONFIG_YAML)
    if key in bundle:
        config = yaml.safe_load(bundle[key])
    else:
        with path.open("r", encoding="utf-8") as handle:
            config = yaml.safe_load(handle)
    if not isinstance(config, dict):
        raise TypeError(f"Configuration must be a mapping: {path}")
    parents = config.pop("extends", [])
    if isinstance(parents, str):
        parents = [parents]
    resolved: dict[str, Any] = {}
    for parent in parents:
        resolved = merge_config(resolved, load_config(path.parent / parent, (*_seen, path)))
    return normalize_config(merge_config(resolved, config))


def file_hash(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def implementation_hash(entrypoint=None) -> str:
    """Hash saved definition cells, excluding outputs, UI cells and configuration."""
    notebook = json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8"))
    payload = [
        (cell["id"], "".join(cell["source"]))
        for cell in notebook["cells"]
        if cell["cell_type"] == "code"
        and "library" in cell.get("metadata", {}).get("tags", [])
        and cell["id"] != "configuration"
    ]
    digest = hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()
    if globals().get("LOADED_IMPLEMENTATION_HASH", digest) != digest:
        raise RuntimeError("Notebook implementation changed: restart and run definition cells")
    return digest


def config_signature(config: dict) -> str:
    payload = {
        key: value
        for key, value in normalize_config(config).items()
        if key not in {"output_dir", "resume"}
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()


def validate_training_config(config: dict, stage: str) -> None:

    if int(config.get("num_workers", 0)) != 0:
        raise ValueError("Notebook data loaders require num_workers=0; cache decoding uses threads")
    required = {
        "seed",
        "encoder",
        "image_size",
        "batch_size",
        "epochs",
        "learning_rate",
        "weight_decay",
        "output_dir",
        "num_workers",
        "resume",
        "target_policy",
        "optimizer",
        "precision",
        "multiprocessing_context",
    }
    required |= (
        {"manifest", "development_manifest", "ssl", "augmentation"}
        if stage == "pretrain"
        else {
            "train_manifest",
            "development_manifest",
            "sampled_patients",
            "mode",
            "label_fraction",
            "early_stopping_patience",
            "supervised_augmentation",
            "imagenet_pretrained",
        }
    )
    missing = required.difference(config)
    if missing:
        raise ValueError(f"Missing {stage} configuration keys: {sorted(missing)}")
    if config["encoder"] not in config.get("architectures", ["resnet18", "resnet50"]):
        raise ValueError("Encoder is not allowed by the shared backbone configuration")
    expected = {label: int(label in UNCERTAIN_AS_POSITIVE) for label in TARGETS}
    if config["target_policy"] != expected:
        raise ValueError("Primary experiments require the declared U-Ones/U-Zeros target policy")
    for key in ("batch_size", "epochs", "image_size", "learning_rate"):
        if not math.isfinite(float(config[key])) or float(config[key]) <= 0:
            raise ValueError(f"{key} must be positive")
    if int(config["num_workers"]) < 0 or float(config["weight_decay"]) < 0:
        raise ValueError("num_workers and weight_decay must be nonnegative")
    if config["optimizer"] != "adamw" or config["precision"] not in {"auto", "float32"}:
        raise ValueError("Supported optimizer/precision: adamw and auto/float32")
    if config["multiprocessing_context"] not in {"spawn", "forkserver"}.intersection(
        multiprocessing.get_all_start_methods()
    ):
        raise ValueError("Use spawn or forkserver for safe training worker creation")
    if stage == "pretrain":
        if config.get("method") not in METHODS or config.get("schedule") != "cosine":
            raise ValueError("Pretraining requires a supported SSL method and cosine schedule")
        if int(config["batch_size"]) < 2:
            raise ValueError("SSL batch size must be at least two")
        if int(config.get("warmup_epochs", 0)) < 0:
            raise ValueError("Warmup epochs must be nonnegative")
        if int(config.get("max_steps_per_epoch", 1)) < 1:
            raise ValueError("max_steps_per_epoch must be positive")
        splits = int(config["ssl"].get("bn_splits", 1))
        if splits < 1 or (config["method"] == "moco" and int(config["batch_size"]) % splits):
            raise ValueError("MoCo batch size must be divisible by positive BatchNorm splits")
    else:
        if config["mode"] not in {
            "supervised",
            "ssl_linear",
            "ssl_finetune",
            "simclr_linear",
            "simclr_finetune",
        }:
            raise ValueError("Unsupported downstream mode")
        if (
            not 0 < float(config["label_fraction"]) <= 1
            or int(config["early_stopping_patience"]) < 1
        ):
            raise ValueError("Invalid label fraction or early-stopping patience")
        if not config["sampled_patients"]:
            raise ValueError("A persisted sampled_patients cohort is required")
        if config.get("pos_weight_max") is not None and float(config["pos_weight_max"]) < 1:
            raise ValueError("pos_weight_max must be at least one")
        if config["mode"] != "supervised":
            if config.get("ssl_method") not in METHODS or not config.get("ssl_checkpoint"):
                raise ValueError("SSL transfer requires ssl_method and ssl_checkpoint")
            if config["imagenet_pretrained"]:
                raise ValueError("SSL and ImageNet initialization cannot be combined")
        if config["imagenet_pretrained"] and not config.get("imagenet_weights"):
            raise ValueError("ImageNet initialization requires an explicit weight variant")


def capture_rng_state() -> dict:
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
    }


def restore_rng_state(state: dict) -> None:
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"].cpu())
    if torch.cuda.is_available() and state["cuda"]:
        torch.cuda.set_rng_state_all([value.cpu() for value in state["cuda"]])


def atomic_torch_save(payload: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    descriptor, temporary = tempfile.mkstemp(dir=path.parent, suffix=".tmp")
    os.close(descriptor)
    try:
        torch.save(payload, temporary)
        os.replace(temporary, path)
    finally:
        Path(temporary).unlink(missing_ok=True)


def save_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    descriptor, temporary = tempfile.mkstemp(dir=path.parent, suffix=".tmp")
    try:
        with os.fdopen(descriptor, "w", encoding="utf-8") as handle:
            json.dump(payload, handle, indent=2, sort_keys=True, default=str)
        os.replace(temporary, path)
    finally:
        Path(temporary).unlink(missing_ok=True)


def run_metadata(config: dict[str, Any], command: list[str] | None = None) -> dict[str, Any]:
    """Common provenance persisted in every output directory."""
    try:
        commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
    except (OSError, subprocess.CalledProcessError):
        commit = "unavailable"
    return {
        "config": config,
        "command": command or globals().get("ACTIVE_TASK_COMMAND") or ["notebook", "direct-call"],
        "git_commit": commit,
        "python": sys.version,
        "torch": torch.__version__,
        "software_versions": {
            name: importlib.metadata.version(name)
            for name in (
                "torch",
                "torchvision",
                "numpy",
                "pandas",
                "Pillow",
                "PyYAML",
                "scikit-learn",
            )
        },
        "cuda_available": torch.cuda.is_available(),
        "hardware": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    }


def save_run_metadata(output_dir: Path, config: dict[str, Any]) -> None:
    path = output_dir / "run_metadata.json"
    metadata = run_metadata(config)
    if path.exists():
        previous = load_config(path)
        if config_signature(previous["config"]) != config_signature(config):
            raise ValueError("Output directory belongs to a different configuration")
        metadata["timestamp_utc"] = previous["timestamp_utc"]
        metadata["resumed_at_utc"] = datetime.now(timezone.utc).isoformat()
    metadata["status"] = "running"
    save_json(path, metadata)
    save_json(output_dir / "resolved_config.json", config)


def finish_run(output_dir: Path, **details: Any) -> None:
    path = output_dir / "run_metadata.json"
    metadata = load_config(path)
    metadata.update(details)
    metadata["status"] = "complete"
    metadata["ended_at_utc"] = datetime.now(timezone.utc).isoformat()
    save_json(path, metadata)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def device_from_config(value: str) -> torch.device:
    if value == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(value)

## Selection

In [90]:
"""Development-only selection records bound to immutable checkpoint artifacts."""


def lock_run(run_dir: Path, reason: str) -> Path:
    """Record a human selection decision without opening held-out records."""
    with artifact_lock(run_dir / ".run.lock"):
        return _lock_run(run_dir, reason)


def _lock_run(run_dir: Path, reason: str) -> Path:
    if not reason.strip():
        raise ValueError("A development-selection reason is required")
    metadata = load_config(run_dir / "run_metadata.json")
    metrics = load_config(run_dir / "metrics.json")
    if metadata.get("status") != "complete" or metrics.get("partition") != "development":
        raise ValueError("Only completed development-selected runs can be locked")
    checkpoint = torch.load(run_dir / "best.pt", map_location="cpu", weights_only=False)
    signature = checkpoint.get("training_signature")
    if (
        not signature
        or signature != config_signature(metadata["config"])
        or signature != metrics.get("training_signature")
    ):
        raise ValueError("Checkpoint, metrics, and metadata configurations disagree")
    thresholds = load_config(run_dir / "thresholds.json")
    if thresholds != checkpoint.get("thresholds"):
        raise ValueError("Thresholds do not belong to the selected checkpoint")
    lock = {
        "selected_on": "development",
        "reason": reason,
        "locked_at_utc": datetime.now(timezone.utc).isoformat(),
        "source_run_id": run_dir.name,
        "training_signature": signature,
        "checkpoint_sha256": file_hash(run_dir / "best.pt"),
        "thresholds_sha256": file_hash(run_dir / "thresholds.json"),
        "metrics_sha256": file_hash(run_dir / "metrics.json"),
    }
    path = run_dir / "selection_lock.json"
    if path.exists():
        raise ValueError("Selection lock already exists; selected runs are immutable")
    save_json(path, lock)
    return path


def verify_selection(config: dict, checkpoint: dict) -> tuple[dict, dict]:
    lock = load_config(Path(config["selection_lock"]))
    training = checkpoint["config"]
    signature = checkpoint.get("training_signature")
    if (
        lock.get("selected_on") != "development"
        or not signature
        or signature != config_signature(training)
    ):
        raise ValueError("A valid development-selection lock is required")
    if lock.get("training_signature") != signature:
        raise ValueError("Selection lock configuration does not match checkpoint")
    for config_key, lock_key in (
        ("checkpoint", "checkpoint_sha256"),
        ("thresholds", "thresholds_sha256"),
    ):
        if file_hash(config[config_key]) != lock.get(lock_key):
            raise ValueError(f"Selected {config_key} artifact changed after locking")
    metrics_path = Path(config["checkpoint"]).parent / "metrics.json"
    if file_hash(metrics_path) != lock.get("metrics_sha256"):
        raise ValueError("Selected development metrics changed after locking")
    for key in ("encoder", "image_size", "target_policy", "supervised_augmentation"):
        if key in config and config[key] != training[key]:
            raise ValueError(f"Evaluation {key} conflicts with selected training configuration")
    thresholds = load_config(Path(config["thresholds"]))
    if thresholds != checkpoint.get("thresholds"):
        raise ValueError("Thresholds do not match selected checkpoint")
    return training, lock

## Experiments

In [91]:
"""Build a reproducible image-only experiment matrix from resolved YAML templates."""


def build_plan(matrix: dict) -> dict[str, list[dict]]:
    if matrix.get("execution_order", "method") not in {"method", "notebook"}:
        raise ValueError("Unknown experiment execution order")
    if set(matrix["pretrain_configs"]) != set(METHODS):
        raise ValueError("The primary matrix must include SimCLR, MoCo, BYOL, NNCLR, and SwAV")
    if set(matrix["downstream_templates"]) != {"ssl_linear", "ssl_finetune", "scratch", "imagenet"}:
        raise ValueError("Matrix must include both SSL protocols and scratch/ImageNet references")
    budgets, seeds = matrix["budgets"], matrix["seeds"]
    if (
        not budgets
        or len(set(budgets)) != len(budgets)
        or any(not 0 < float(b) <= 1 for b in budgets)
    ):
        raise ValueError("Budgets must be unique fractions in (0, 1]")
    if not seeds or len(set(seeds)) != len(seeds) or any(not isinstance(s, int) for s in seeds):
        raise ValueError("Seeds must be unique integers")
    pretrain = []
    shared_pretrain = (
        "seed",
        "encoder",
        "image_size",
        "batch_size",
        "epochs",
        "manifest",
        "development_manifest",
        "augmentation",
        "optimizer",
        "learning_rate",
        "weight_decay",
        "schedule",
        "precision",
        "target_policy",
        "cache_path",
    )
    for method in METHODS:
        config = load_config(Path(matrix["pretrain_configs"][method]))
        config = merge_config(config, matrix.get("pretrain_overrides", {}))
        config = merge_config(config, matrix.get("method_overrides", {}).get(method, {}))
        if matrix.get("pretrain_output_root"):
            config["output_dir"] = str(
                Path(matrix["pretrain_output_root"])
                / f"{method}-{config['encoder']}-seed{config['seed']}"
            )
        validate_training_config(config, "pretrain")
        if config["method"] != method:
            raise ValueError("Pretraining config method does not match matrix arm")
        if pretrain and any(config.get(key) != pretrain[0].get(key) for key in shared_pretrain):
            raise ValueError(
                "All five pretraining methods must share the primary comparison settings"
            )
        pretrain.append(config)
    templates = {
        key: merge_config(load_config(Path(path)), matrix.get("downstream_overrides", {}))
        for key, path in matrix["downstream_templates"].items()
    }
    common = (
        "encoder",
        "image_size",
        "train_manifest",
        "development_manifest",
        "target_policy",
        "batch_size",
        "epochs",
        "early_stopping_patience",
        "supervised_augmentation",
        "precision",
    )
    reference = templates["scratch"]
    for config in templates.values():
        if any(config.get(key) != reference.get(key) for key in common):
            raise ValueError("Downstream arms must share data and evaluation settings")
        if any(
            config[key] != pretrain[0][key] for key in ("encoder", "image_size", "target_policy")
        ):
            raise ValueError(
                "Downstream architecture, resolution, and targets must match pretraining"
            )
    downstream, evaluation = [], []
    for budget in sorted(budgets):
        fraction = float(budget)
        token = f"{fraction:g}".replace(".", "p")
        for seed in seeds:
            cohort = str(Path(matrix["splits_dir"]) / f"budget_{token}_seed_{seed}.json")
            arms = [
                (method, protocol)
                for method in METHODS
                for protocol in ("ssl_linear", "ssl_finetune")
            ]
            arms += [("scratch", "scratch"), ("imagenet", "imagenet")]
            if matrix.get("execution_order") == "notebook":
                arms = [("scratch", "scratch"), ("imagenet", "imagenet")]
                arms += [
                    (method, protocol)
                    for protocol in ("ssl_linear", "ssl_finetune")
                    for method in METHODS
                ]
            for method, protocol in arms:
                config = dict(templates[protocol])
                config = merge_config(
                    config, matrix.get("protocol_overrides", {}).get(protocol, {})
                )
                if protocol != "ssl_linear" and matrix.get("epochs_by_budget"):
                    config["epochs"] = matrix["epochs_by_budget"][fraction]
                config.update(seed=seed, label_fraction=fraction, sampled_patients=cohort)
                if method in METHODS:
                    source = next(c for c in pretrain if c["method"] == method)
                    config.update(
                        mode=protocol,
                        ssl_method=method,
                        ssl_checkpoint=str(
                            Path(source["output_dir"])
                            / matrix.get("transfer_checkpoint", "best.pt")
                        ),
                    )
                else:
                    config.pop("ssl_method", None)
                    config.pop("ssl_checkpoint", None)
                    config.update(mode="supervised", imagenet_pretrained=method == "imagenet")
                run_id = f"{method}-{protocol}-{config['encoder']}-budget{token}-seed{seed}"
                config["output_dir"] = str(Path(matrix["downstream_output_root"]) / run_id)
                validate_training_config(config, "downstream")
                downstream.append(config)
                evaluation.append(
                    {
                        "device": config["device"],
                        "seed": seed,
                        "checkpoint": str(Path(config["output_dir"]) / "best.pt"),
                        "thresholds": str(Path(config["output_dir"]) / "thresholds.json"),
                        "selection_lock": str(Path(config["output_dir"]) / "selection_lock.json"),
                        "manifest": matrix["final_manifest"],
                        "batch_size": config["batch_size"],
                        "num_workers": config["num_workers"],
                        "bootstrap_samples": matrix["bootstrap_samples"],
                        **(
                            {"per_label_bootstrap": True}
                            if matrix.get("per_label_bootstrap")
                            else {}
                        ),
                        "output_dir": str(Path(matrix["evaluation_output_root"]) / run_id),
                    }
                )
    directories = [c["output_dir"] for c in (*pretrain, *downstream, *evaluation)]
    if len(set(directories)) != len(directories):
        raise ValueError("Experiment output directories must be unique")
    return {"pretrain": pretrain, "downstream": downstream, "evaluate": evaluation}

## Vlm

Optional radiology image-text adapter. Disabled until provenance and license review are supplied; this extension remains experimental.

In [92]:
"""Optional Hugging Face VLM adapters and deterministic prompt scoring."""


def _transformers() -> tuple[Any, Any]:
    try:
        from transformers import AutoModel, AutoProcessor
    except ImportError as error:  # pragma: no cover - exercised only in VLM environments
        raise RuntimeError("Install VLM dependencies with: pip install -e '.[vlm]'") from error
    return AutoModel, AutoProcessor


def validate_vlm_config(config):
    """Require explicit reviewed radiology provenance before loading model weights."""
    if config.get("pretraining_modality") != "image-text VLM":
        raise ValueError("VLM reporting requires pretraining_modality: image-text VLM")
    for key in ("checkpoint", "revision", "model_card_url", "license_review", "overlap_review"):
        value = config.get(key)
        if (
            not isinstance(value, str)
            or not value.strip()
            or "REPLACE" in value
            or "required-before" in value
        ):
            raise ValueError(f"Complete VLM provenance field: {key}")
    if not re.fullmatch(r"[0-9a-fA-F]{40}", config["revision"]):
        raise ValueError("VLM revision must be an immutable 40-character commit hash")
    if not config["model_card_url"].startswith("https://"):
        raise ValueError("VLM model_card_url must be an HTTPS source")
    if config.get("radiology_domain_reviewed") is not True:
        raise ValueError("Explicit radiology_domain_reviewed: true is required")


class HuggingFaceVLM(nn.Module):
    """Small adapter for CLIP-compatible radiology checkpoints such as CheXzero."""

    def __init__(self, checkpoint: str, revision: str | None = None) -> None:
        super().__init__()
        auto_model, auto_processor = _transformers()
        self.model = auto_model.from_pretrained(checkpoint, revision=revision)
        self.processor = auto_processor.from_pretrained(checkpoint, revision=revision)
        self.feature_dim = int(getattr(self.model.config, "projection_dim", 512))

    def image_features(self, pixel_values: torch.Tensor) -> torch.Tensor:
        if not hasattr(self.model, "get_image_features"):
            raise ValueError("Checkpoint must expose get_image_features for this VLM adapter")
        return self.model.get_image_features(pixel_values=pixel_values)

    @torch.inference_mode()
    def text_features(self, prompts: list[str], device: torch.device) -> torch.Tensor:
        batch = self.processor(text=prompts, return_tensors="pt", padding=True, truncation=True)
        batch = {key: value.to(device) for key, value in batch.items()}
        if not hasattr(self.model, "get_text_features"):
            raise ValueError("Checkpoint must expose get_text_features for zero-shot scoring")
        return F.normalize(self.model.get_text_features(**batch), dim=-1)


class VLMClassifier(nn.Module):
    def __init__(self, encoder: HuggingFaceVLM, num_labels: int, freeze_encoder: bool) -> None:
        super().__init__()
        self.encoder = encoder
        self.freeze_encoder = freeze_encoder
        self.classifier = nn.Linear(encoder.feature_dim, num_labels)
        if freeze_encoder:
            for parameter in self.encoder.parameters():
                parameter.requires_grad = False

    def train(self, mode=True):
        super().train(mode)
        if self.freeze_encoder:
            self.encoder.eval()
        return self

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.encoder.image_features(pixel_values))


def aggregate_prompt_embeddings(
    model: HuggingFaceVLM, prompt_set: dict[str, dict[str, list[str]]], device: torch.device
) -> dict[str, tuple[torch.Tensor, torch.Tensor]]:
    """Average normalized template embeddings for each positive/negative concept."""
    result: dict[str, tuple[torch.Tensor, torch.Tensor]] = {}
    for label in TARGETS:
        specification = prompt_set[label]
        positive = F.normalize(
            model.text_features(specification["positive"], device).mean(dim=0), dim=0
        )
        negative = F.normalize(
            model.text_features(specification["negative"], device).mean(dim=0), dim=0
        )
        result[label] = (positive, negative)
    return result


def zero_shot_scores(
    image_embeddings: torch.Tensor, concepts: dict[str, tuple[torch.Tensor, torch.Tensor]]
) -> np.ndarray:
    """Two-class softmax probability for each pathology; labels never enter this function."""
    images = F.normalize(image_embeddings, dim=-1)
    scores = []
    for label in TARGETS:
        positive, negative = concepts[label]
        logits = torch.stack((images @ negative, images @ positive), dim=-1)
        scores.append(torch.softmax(logits, dim=-1)[:, 1])
    return torch.stack(scores, dim=1).cpu().numpy()

## Prepare Data

In [93]:
"""Create portable CheXpert manifests and patient-disjoint development splits."""


def parse_prepare_data_args(argv=None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", type=Path)
    parser.add_argument("--dataset-root", type=Path)
    parser.add_argument("--output-dir", type=Path, default=Path("data/processed/manifests"))
    parser.add_argument("--dev-fraction", type=float, default=0.1)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--frontal-only-downstream", action="store_true")
    parser.add_argument("--budgets", type=float, nargs="+", default=[0.01, 0.05, 0.1, 0.25, 1.0])
    parser.add_argument("--budget-seeds", type=int, nargs="+", default=[42, 43, 44])
    parser.add_argument("--random-state", action="store_true")
    parser.add_argument("--budget-rounding", choices=("round", "ceil"), default="round")
    parser.add_argument(
        "--overwrite", action="store_true", help="Explicitly regenerate all manifests and cohorts"
    )
    initial, _ = parser.parse_known_args(argv)
    if initial.config:
        config = load_config(initial.config)
        unknown = set(config).difference({action.dest for action in parser._actions})
        if unknown:
            parser.error(f"Unknown data configuration keys: {sorted(unknown)}")
        parser.set_defaults(**config)
    args = parser.parse_args(argv)
    if not args.dataset_root:
        parser.error("--dataset-root or a data configuration is required")
    if not 0 < args.dev_fraction < 1 or any(not 0 < fraction <= 1 for fraction in args.budgets):
        parser.error("Invalid development fraction or label budget")
    return args


def prepare_data_cli(argv=None) -> None:
    args = parse_prepare_data_args(argv)
    if not args.overwrite and (args.output_dir / "data_report.json").exists():
        raise ValueError(
            "Prepared data already exists; reuse the cohorts or explicitly pass --overwrite"
        )
    root = args.dataset_root
    train = prepare_manifest(root / "train.csv", root, frontal_only=False)
    valid = prepare_manifest(root / "valid.csv", root, frontal_only=args.frontal_only_downstream)
    train, missing_train = existing_images(train)
    valid, missing_valid = existing_images(valid)
    if args.frontal_only_downstream:
        valid = valid.loc[valid["AP/PA"].isin(["AP", "PA"])].copy()
    pretrain_train, development_source = split_by_patient(
        train, args.dev_fraction, args.seed, args.random_state
    )
    downstream_train, dev = pretrain_train, development_source
    if args.frontal_only_downstream:
        downstream_train = pretrain_train.loc[
            pretrain_train["Frontal/Lateral"].eq("Frontal")
            & pretrain_train["AP/PA"].isin(["AP", "PA"])
        ].copy()
        dev = development_source.loc[
            development_source["Frontal/Lateral"].eq("Frontal")
            & development_source["AP/PA"].isin(["AP", "PA"])
        ].copy()
    if downstream_train.empty or dev.empty:
        raise ValueError("View filtering produced an empty train/development partition")
    assert_patient_disjoint(downstream_train, dev, valid)
    assert_patient_disjoint(pretrain_train, development_source, valid)

    args.output_dir.mkdir(parents=True, exist_ok=True)
    splits_dir = args.output_dir.parent / "splits"
    splits_dir.mkdir(parents=True, exist_ok=True)
    pretrain_train.to_csv(args.output_dir / "pretrain_train.csv", index=False)
    train.to_csv(args.output_dir / "all_training.csv", index=False)
    pretrain_train.to_csv(args.output_dir / "pretrain_train_leakage_free.csv", index=False)
    downstream_train.to_csv(args.output_dir / "downstream_train.csv", index=False)
    dev.to_csv(args.output_dir / "development.csv", index=False)
    valid.to_csv(args.output_dir / "final_validation.csv", index=False)
    missing_train.to_csv(args.output_dir / "missing_pretrain_images.csv", index=False)
    missing_valid.to_csv(args.output_dir / "missing_validation_images.csv", index=False)
    pretrain_train[["patient_id"]].drop_duplicates().to_csv(
        splits_dir / "train_patients.csv", index=False
    )
    development_source[["patient_id"]].drop_duplicates().to_csv(
        splits_dir / "development_patients.csv", index=False
    )
    valid[["patient_id"]].drop_duplicates().to_csv(
        splits_dir / "final_validation_patients.csv", index=False
    )
    for budget_seed in args.budget_seeds:
        for fraction, patients in nested_patient_ids(
            downstream_train, args.budgets, budget_seed, args.random_state, args.budget_rounding
        ).items():
            label = f"{fraction:g}".replace(".", "p")
            save_json(
                splits_dir / f"budget_{label}_seed_{budget_seed}.json",
                {
                    "seed": budget_seed,
                    "label_fraction": fraction,
                    "patient_ids": patients,
                    "budget_rounding": args.budget_rounding,
                },
            )
    save_json(
        args.output_dir / "data_report.json",
        {
            "dataset_root": str(root),
            "pretrain_records": len(pretrain_train),
            "downstream_train_records": len(downstream_train),
            "development_records": len(dev),
            "final_validation_records": len(valid),
            "missing_pretrain_images": len(missing_train),
            "missing_validation_images": len(missing_valid),
            "dev_fraction": args.dev_fraction,
            "seed": args.seed,
            "random_state": args.random_state,
            "budget_rounding": args.budget_rounding,
            "frontal_only_downstream": args.frontal_only_downstream,
            "downstream_filtered_records": len(train) - len(downstream_train) - len(dev),
            "manifest_hashes": {
                "pretrain_train": manifest_hash(pretrain_train),
                "downstream_train": manifest_hash(downstream_train),
                "development": manifest_hash(dev),
                "final_validation": manifest_hash(valid),
            },
            "prevalence": {
                "downstream_train": prevalence(downstream_train),
                "development": prevalence(dev),
            },
            "uncertainty_as_positive": ["Atelectasis", "Edema"],
            "budget_seeds": args.budget_seeds,
            "budgets": args.budgets,
        },
    )
    print(f"Prepared manifests in {args.output_dir}")

## Repair Pretrain Manifest

In [94]:
"""Create a development-patient-free pretraining manifest without reading images."""


def parse_repair_pretrain_manifest_args(argv=None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--pretrain-manifest",
        type=Path,
        default=Path("data/processed/manifests/pretrain_train.csv"),
    )
    parser.add_argument(
        "--development-manifest",
        type=Path,
        default=Path("data/processed/manifests/development.csv"),
    )
    parser.add_argument(
        "--output",
        type=Path,
        default=Path("data/processed/manifests/pretrain_train_leakage_free.csv"),
    )
    return parser.parse_args(argv)


def repair_pretrain_manifest_cli(argv=None) -> None:
    args = parse_repair_pretrain_manifest_args(argv)
    for source in (args.pretrain_manifest, args.development_manifest):
        if not source.is_file():
            raise FileNotFoundError(f"Manifest not found: {source}")
    if args.output.resolve() in {
        args.pretrain_manifest.resolve(),
        args.development_manifest.resolve(),
    }:
        raise ValueError("Output must be a separate file; source manifests will not be overwritten")

    pretrain = read_manifest(args.pretrain_manifest, labeled=False)
    development = read_manifest(args.development_manifest, labeled=False)
    reject_final_partition(args.pretrain_manifest, pretrain)
    reject_final_partition(args.development_manifest, development)
    repaired, removed_patients = exclude_development_patients(pretrain, development)
    args.output.parent.mkdir(parents=True, exist_ok=True)
    repaired.to_csv(args.output, index=False)
    report_path = args.output.with_name(f"{args.output.stem}_repair_report.json")
    save_json(
        report_path,
        {
            "source_manifest": str(args.pretrain_manifest),
            "development_manifest": str(args.development_manifest),
            "output_manifest": str(args.output),
            "source_records": len(pretrain),
            "output_records": len(repaired),
            "removed_records": len(pretrain) - len(repaired),
            "removed_development_patients": removed_patients,
            "source_manifest_hash": manifest_hash(pretrain),
            "output_manifest_hash": manifest_hash(repaired),
        },
    )
    print(
        f"Wrote {len(repaired):,} pretraining records to {args.output}; "
        f"excluded {removed_patients:,} development patients ({len(pretrain) - len(repaired):,} records). "
        f"Report: {report_path}"
    )

## Prepare Smoke Manifest

In [95]:
"""Create a small image-only manifest for checking the SimCLR pipeline."""


def parse_prepare_smoke_manifest_args(argv=None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--source",
        type=Path,
        default=Path("data/processed/manifests/pretrain_train_leakage_free.csv"),
    )
    parser.add_argument(
        "--output",
        type=Path,
        default=Path("data/processed/manifests/pretrain_smoke.csv"),
    )
    parser.add_argument("--records", type=int, default=256)
    return parser.parse_args(argv)


def prepare_smoke_manifest_cli(argv=None) -> None:
    args = parse_prepare_smoke_manifest_args(argv)
    if args.records < 1:
        raise ValueError("--records must be at least 1")
    if not args.source.is_file():
        raise FileNotFoundError(f"Prepared pretraining manifest not found: {args.source}")

    frame = read_manifest(args.source, labeled=False)
    reject_final_partition(args.source, frame)
    if frame.empty:
        raise ValueError(f"Prepared pretraining manifest is empty: {args.source}")
    smoke_frame = frame.head(args.records)
    args.output.parent.mkdir(parents=True, exist_ok=True)
    smoke_frame.to_csv(args.output, index=False)
    print(f"Wrote {len(smoke_frame)} records to {args.output}")

## Prepare Downstream Smoke

In [96]:
"""Create a bounded patient-safe downstream smoke from a persisted training cohort."""


def prepare_downstream_smoke_cli(argv=None) -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--config", type=Path, default=Path("configs/downstream/simclr_linear.yaml")
    )
    parser.add_argument("--output-dir", type=Path, default=Path("data/processed/smoke"))
    parser.add_argument("--train-patients", type=int, default=8)
    parser.add_argument("--development-patients", type=int, default=32)
    args = parser.parse_args(argv)
    if min(args.train_patients, args.development_patients) < 1:
        parser.error("Smoke patient limits must be positive")
    config = load_config(args.config)
    train = read_manifest(config["train_manifest"])
    dev = read_manifest(config["development_manifest"])
    reject_final_partition(config["train_manifest"], train)
    reject_final_partition(config["development_manifest"], dev)
    assert_patient_disjoint(train, dev)
    train = select_patient_cohort(
        train, config["sampled_patients"], config["label_fraction"], config["seed"]
    )
    patients = sorted(train.patient_id.unique())[: args.train_patients]
    train = train.loc[train.patient_id.isin(patients)]
    dev = dev.loc[dev.patient_id.isin(sorted(dev.patient_id.unique())[: args.development_patients])]
    args.output_dir.mkdir(parents=True, exist_ok=True)
    train.to_csv(args.output_dir / "downstream_train.csv", index=False)
    dev.to_csv(args.output_dir / "development.csv", index=False)
    import pandas as pd

    pd.concat([train, dev], ignore_index=True).to_csv(
        args.output_dir / "all_training.csv", index=False
    )
    save_json(
        args.output_dir / "cohort.json",
        {"seed": config["seed"], "label_fraction": 1.0, "patient_ids": patients},
    )
    save_json(
        args.output_dir / "provenance.json",
        {
            "source_cohort": config["sampled_patients"],
            "source_cohort_sha256": file_hash(config["sampled_patients"]),
            "train_records": len(train),
            "development_records": len(dev),
        },
    )

## Train Pretrain

All five objectives retain resumable optimizer/scaler/RNG state and feature-spread monitoring.

In [97]:
"""Train one SSL method with a reusable image cache and device-side augmentation."""


def parse_train_pretrain_args(argv=None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", type=Path, required=True)
    return parser.parse_args(argv)


def training_signature(config: dict, method: str, cache_metadata: dict) -> str:
    return config_signature({**config, "method": method, "cache": cache_metadata})


def notebook_learning_rate(step: int, total: int, warm: int, base: float) -> float:
    if step < warm:
        return base * (step + 1) / max(1, warm)
    progress = (step - warm) / max(1, total - warm)
    return base * (0.001 + 0.999 * 0.5 * (1 + math.cos(math.pi * progress)))


@torch.no_grad()
def feature_spread(encoder, dataset, size, normalization, device, sample_count=512):
    encoder.eval()
    indices = np.sort(
        np.random.RandomState(0).choice(
            len(dataset), min(sample_count, len(dataset)), replace=False
        )
    )
    images = torch.from_numpy(np.stack([dataset[int(i)] for i in indices])).to(device)
    inputs = images.float().div(255).unsqueeze(1)
    inputs = (inputs - normalization["mean"]) / normalization["std"]
    features = torch.nn.functional.normalize(encoder(inputs.expand(-1, 3, -1, -1)).float(), dim=1)
    encoder.train()
    return float(features.std(dim=0).mean() * math.sqrt(features.shape[1]))


def save_augmentation_preview(
    dataset,
    image_size: int,
    augmentation: dict,
    device: torch.device,
    channels_last: bool,
    output_path: Path,
    seed: int,
) -> None:
    count = min(8, len(dataset))
    indices = np.sort(np.random.default_rng(seed).choice(len(dataset), count, replace=False))
    grayscale = torch.from_numpy(np.stack([dataset[int(index)] for index in indices])).to(device)
    normalization = augmentation.get("normalization", {"mean": 0.5, "std": 0.5})
    original = grayscale.float().div(255).sub(normalization["mean"]).div(normalization["std"])
    original = original.unsqueeze(1).expand(-1, 3, -1, -1)
    first = augment_grayscale_batch(grayscale, image_size, augmentation, channels_last)
    second = augment_grayscale_batch(grayscale, image_size, augmentation, channels_last)
    preview = torch.cat((original, first, second)).cpu()
    save_image(
        preview,
        output_path,
        normalize=True,
        value_range=(
            -normalization["mean"] / normalization["std"],
            (1 - normalization["mean"]) / normalization["std"],
        ),
        nrow=count,
    )


@managed_run
def train_pretrain(config: dict) -> None:
    config = dict(config)
    validate_training_config(config, "pretrain")
    method_name = str(config.get("method", "simclr")).lower()
    if method_name not in METHODS:
        raise ValueError(f"method must be one of {', '.join(METHODS)}")

    seed_everything(int(config["seed"]))
    device = device_from_config(config.get("device", "auto"))
    amp = device.type == "cuda" and config["precision"] == "auto"
    config["effective_precision"] = "float16" if amp else "float32"
    torch.set_num_threads(int(config.get("torch_num_threads", 4)))
    torch.backends.cudnn.deterministic = bool(config.get("deterministic", False))
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = bool(config.get("cudnn_benchmark", True))
        torch.backends.cudnn.allow_tf32 = bool(config.get("allow_tf32", True))
        torch.backends.cuda.matmul.allow_tf32 = bool(config.get("allow_tf32", True))
        torch.set_float32_matmul_precision("high")

    output_dir = Path(config["output_dir"])
    if not (output_dir / "last.pt").exists() and any(
        (output_dir / name).exists() for name in ("best.pt", "metrics.json")
    ):
        raise ValueError("Existing run artifacts without a resumable checkpoint")
    if not config["resume"] and any(
        (output_dir / name).exists() for name in ("last.pt", "best.pt", "metrics.json")
    ):
        raise ValueError("Output directory already contains training artifacts")
    frame = read_manifest(config["manifest"], labeled=False)
    development = read_manifest(config["development_manifest"], labeled=False)
    reject_final_partition(config["manifest"], frame)
    reject_final_partition(config["development_manifest"], development)
    assert_patient_disjoint(frame, development)
    config["identifier_normalization"] = {
        "manifest_restored_rows": frame.attrs.get("patient_id_padding_restored_rows", 0),
        "development_restored_rows": development.attrs.get("patient_id_padding_restored_rows", 0),
    }
    config["provenance"] = {
        key: file_hash(config[key]) for key in ("manifest", "development_manifest")
    }
    if len(frame) < int(config["batch_size"]):
        raise ValueError("Pretraining manifest must contain at least one full batch")

    image_size = int(config["image_size"])
    cache_path = Path(
        config.get("cache_path", f"data/processed/cache/pretrain_gray_{image_size}.npy")
    )
    cache_frame = frame
    if config.get("cache_source_manifest"):
        cache_frame = read_manifest(config["cache_source_manifest"], labeled=False)
        reject_final_partition(config["cache_source_manifest"], cache_frame)
        config["provenance"]["cache_source_manifest"] = file_hash(config["cache_source_manifest"])
    with artifact_lock(cache_path.with_suffix(".lock")):
        dataset, cache_built = ensure_grayscale_cache(
            cache_frame,
            cache_path,
            image_size,
            workers=int(config.get("cache_workers", 16)),
            log_interval=int(config.get("cache_log_interval", 10_000)),
        )
        cache_metadata = json.loads(cache_path.with_suffix(".json").read_text(encoding="utf-8"))
    if config.get("cache_source_manifest"):
        lookup = {value: index for index, value in enumerate(cache_frame.image_path)}
        dataset = torch.utils.data.Subset(dataset, [lookup[value] for value in frame.image_path])
    signature = training_signature(config, method_name, cache_metadata)

    if device.type == "cuda":
        print(
            f"device=cuda GPU={torch.cuda.get_device_name(device)} "
            f"VRAM_GB={torch.cuda.get_device_properties(device).total_memory / 2**30:.1f}",
            flush=True,
        )
    else:
        print("device=cpu; CUDA is unavailable", flush=True)
    print(
        f"method={method_name} images={len(dataset):,} image_size={image_size} "
        f"batch_size={config['batch_size']} cache={'built' if cache_built else 'reused'} "
        f"cache_path={cache_path}",
        flush=True,
    )

    augmentation = config.get("augmentation", {})
    channels_last = bool(config.get("channels_last", True)) and device.type == "cuda"

    workers = int(config.get("num_workers", 8))
    shuffle_generator = torch.Generator()
    worker_generator = torch.Generator()
    loader_options = {
        "dataset": dataset,
        "batch_size": int(config["batch_size"]),
        "sampler": RandomSampler(dataset, generator=shuffle_generator),
        "generator": worker_generator,
        "num_workers": workers,
        "pin_memory": device.type == "cuda",
        "drop_last": True,
        "persistent_workers": workers > 0,
    }
    if workers > 0:
        loader_options["prefetch_factor"] = int(config.get("prefetch_factor", 4))
        loader_options["multiprocessing_context"] = config["multiprocessing_context"]
    batch_sampler = None
    if config.get("notebook_sampling"):
        batch_sampler = NotebookBatchSampler(
            len(dataset), int(config["batch_size"]), int(config["seed"]), "ssl"
        )
        for key in ("batch_size", "sampler", "drop_last"):
            loader_options.pop(key)
        loader_options["batch_sampler"] = batch_sampler
    loader = DataLoader(**loader_options)
    steps_per_epoch = min(len(loader), int(config.get("max_steps_per_epoch", len(loader))))

    model = build_ssl_method(method_name, config.get("encoder", "resnet18"), config.get("ssl", {}))
    if method_name == "swav" and config["ssl"].get("prototype_freeze_epochs") is not None:
        model.freeze_prototype_steps = (
            int(config["ssl"]["prototype_freeze_epochs"]) * steps_per_epoch
        )
    if channels_last:
        model = model.to(device=device, memory_format=torch.channels_last)
    else:
        model = model.to(device)
    fused = bool(config.get("fused_optimizer", True)) and device.type == "cuda"
    optimizer = AdamW(
        (parameter for parameter in model.parameters() if parameter.requires_grad),
        lr=float(config["learning_rate"]),
        weight_decay=float(config["weight_decay"]),
        fused=fused,
    )
    scheduler = CosineAnnealingLR(optimizer, T_max=int(config["epochs"]))
    notebook_schedule = bool(config.get("notebook_schedule", False))
    base_lr = float(config["learning_rate"]) * (
        int(config["batch_size"]) / 256 if notebook_schedule else 1
    )
    scaler = torch.amp.GradScaler("cuda", enabled=amp)

    start_epoch = 1
    best_loss = float("inf")
    history: list[dict[str, float]] = []
    resume_path = output_dir / "last.pt"
    if bool(config.get("resume", True)) and resume_path.is_file():
        checkpoint = torch.load(resume_path, map_location=device, weights_only=False)
        if checkpoint.get("training_signature") != signature:
            raise ValueError(
                "Checkpoint settings do not match this optimized training configuration. "
                "Use a new output_dir or restore the exact settings used to create it."
            )
        model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        scheduler.load_state_dict(checkpoint["scheduler"])
        scaler.load_state_dict(checkpoint["scaler"])
        start_epoch = int(checkpoint["epoch"]) + 1
        best_loss = float(checkpoint["best_loss"])
        history = checkpoint["history"]
        restore_rng_state(checkpoint["rng_state"])
        print(f"Resuming {method_name} at epoch {start_epoch}/{config['epochs']}", flush=True)

    save_run_metadata(output_dir, config)
    if not history:
        preview_rng = capture_rng_state()
        save_augmentation_preview(
            dataset,
            image_size,
            augmentation,
            device,
            channels_last,
            output_dir / "augmentation_pairs.png",
            int(config["seed"]),
        )
        if config.get("notebook_sampling"):
            restore_rng_state(preview_rng)
    print(
        f"mixed_precision={amp} channels_last={channels_last} "
        f"workers={workers} prefetch_factor={config.get('prefetch_factor', 4)} "
        f"fused_optimizer={fused}",
        flush=True,
    )

    for epoch in range(start_epoch, int(config["epochs"]) + 1):
        if batch_sampler is not None:
            batch_sampler.epoch = epoch - 1
        shuffle_generator.manual_seed(int(config["seed"]) + epoch)
        worker_generator.manual_seed(int(config["seed"]) + epoch)
        epoch_started = time.perf_counter()
        if device.type == "cuda":
            torch.cuda.reset_peak_memory_stats(device)
        model.train()
        total_loss = torch.zeros((), device=device, dtype=torch.float32)
        log_every = max(1, steps_per_epoch // 10)
        interval_started = time.perf_counter()
        last_logged_step = 0
        for batch_index, grayscale in enumerate(loader, start=1):
            if batch_index > steps_per_epoch:
                break
            step = (epoch - 1) * steps_per_epoch + batch_index - 1
            if notebook_schedule:
                for group in optimizer.param_groups:
                    group["lr"] = notebook_learning_rate(
                        step,
                        int(config["epochs"]) * steps_per_epoch,
                        int(config.get("warmup_epochs", 2)) * steps_per_epoch,
                        base_lr,
                    )
            grayscale = grayscale.to(device, non_blocking=True)
            first = augment_grayscale_batch(grayscale, image_size, augmentation, channels_last)
            second = augment_grayscale_batch(grayscale, image_size, augmentation, channels_last)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=amp):
                loss = model(first, second)
            if not torch.isfinite(loss):
                raise FloatingPointError(
                    f"Nonfinite {method_name} loss at epoch {epoch}, step {batch_index}"
                )
            scaler.scale(loss).backward()
            previous_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() >= previous_scale:
                model.after_optimizer_step(step / (int(config["epochs"]) * steps_per_epoch))
            total_loss.add_(loss.detach().float())
            if batch_index % log_every == 0 or batch_index == steps_per_epoch:
                if device.type == "cuda":
                    allocated_gb = torch.cuda.memory_allocated(device) / 2**30
                else:
                    allocated_gb = 0.0
                reported_loss = float(loss.detach().item())
                interval_seconds = time.perf_counter() - interval_started
                interval_steps = batch_index - last_logged_step
                print(
                    f"method={method_name} epoch={epoch} step={batch_index}/{steps_per_epoch} "
                    f"loss={reported_loss:.4f} steps_per_sec={interval_steps / interval_seconds:.2f} "
                    f"gpu_memory_gb={allocated_gb:.2f}",
                    flush=True,
                )
                last_logged_step = batch_index
                interval_started = time.perf_counter()

        if not notebook_schedule:
            scheduler.step()
        mean_loss = float((total_loss / max(1, steps_per_epoch)).item())
        if device.type == "cuda":
            torch.cuda.synchronize(device)
            peak_memory_gb = torch.cuda.max_memory_allocated(device) / 2**30
        else:
            peak_memory_gb = 0.0
        epoch_seconds = time.perf_counter() - epoch_started
        images_per_second = steps_per_epoch * int(config["batch_size"]) / max(epoch_seconds, 1e-9)
        history.append(
            {
                "epoch": epoch,
                "train_loss": mean_loss,
                "learning_rate": optimizer.param_groups[0]["lr"],
                "seconds": epoch_seconds,
                "images_per_second": images_per_second,
                "peak_gpu_memory_gb": peak_memory_gb,
            }
        )
        if config.get("monitor_feature_spread"):
            history[-1]["feature_spread"] = feature_spread(
                model.encoder,
                dataset,
                image_size,
                augmentation.get("normalization", {"mean": 0.5, "std": 0.5}),
                device,
                int(config.get("feature_spread_samples", 512)),
            )
        print(
            f"method={method_name} epoch={epoch} train_loss={mean_loss:.4f} "
            f"seconds={epoch_seconds:.1f} images_per_second={images_per_second:.1f} "
            f"peak_gpu_memory_gb={peak_memory_gb:.2f}",
            flush=True,
        )

        checkpoint_data = {
            "model": model.state_dict(),
            "method": method_name,
            "encoder": encoder_state_dict(model),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
            "config": config,
            "training_signature": signature,
            "epoch": epoch,
            "best_loss": best_loss,
            "history": history,
            "rng_state": capture_rng_state(),
        }
        if mean_loss < best_loss:
            best_loss = mean_loss
            checkpoint_data["best_loss"] = best_loss
            atomic_torch_save(checkpoint_data, output_dir / "best.pt")
        atomic_torch_save(checkpoint_data, resume_path)

    save_json(
        output_dir / "metrics.json",
        {"method": method_name, "history": history, "device": str(device)},
    )
    finish_run(
        output_dir,
        epochs_completed=history[-1]["epoch"],
        training_seconds=sum(row["seconds"] for row in history),
        checkpoint_path=str(output_dir / "last.pt"),
        checkpoint_bytes=resume_path.stat().st_size,
    )


def train_pretrain_cli(argv=None) -> None:
    train_pretrain(load_config(parse_train_pretrain_args(argv).config))

## Train Downstream

Shared patient budgets, development-only selection and thresholds, frozen feature standardization, and resumable fine-tuning.

In [98]:
"""Train matched supervised, SSL frozen-probe, and SSL fine-tuning experiments."""


def positive_weights(frame: pd.DataFrame, maximum: float | None = None) -> torch.Tensor:
    targets = frame[target_columns()].to_numpy(dtype=np.float32)
    positives = targets.sum(axis=0)
    weights = (len(targets) - positives) / np.clip(positives, 1, None)
    if maximum is not None:
        weights = np.clip(weights, 1.0, maximum)
    return torch.tensor(weights, dtype=torch.float32)


def prepare_cached_inputs(inputs, normalization):
    inputs = inputs.float().div(255).unsqueeze(1)
    return ((inputs - normalization["mean"]) / normalization["std"]).expand(-1, 3, -1, -1)


@torch.inference_mode()
def predict(
    model: nn.Module, loader: DataLoader, device: torch.device, normalization=None, amp=False
) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    truths, scores = [], []
    for inputs, targets in loader:
        truths.append(targets.numpy())
        inputs = inputs.to(device)
        if normalization and inputs.dtype == torch.uint8:
            inputs = prepare_cached_inputs(inputs, normalization)
        with torch.autocast(device_type=device.type, enabled=amp):
            logits = model(inputs)
        scores.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(truths), np.concatenate(scores)


def extract_features(
    model: MultiLabelClassifier,
    frame: pd.DataFrame,
    image_size: int,
    batch_size: int,
    workers: int,
    device: torch.device,
    multiprocessing_context: str = "spawn",
    config: dict | None = None,
) -> TensorDataset:
    config = config or {}
    dataset = (
        CachedTargetDataset(
            frame,
            config["downstream_cache_path"],
            image_size,
            config.get("training_cache_manifest"),
        )
        if config.get("cached_downstream")
        else CheXpertDataset(
            frame, supervised_transform(image_size, False, config.get("supervised_augmentation"))
        )
    )
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=workers,
        multiprocessing_context=multiprocessing_context if workers else None,
        generator=torch.Generator().manual_seed(0),
    )
    model.encoder.eval()
    features, labels = [], []
    for images, targets in loader:
        with torch.no_grad():
            images = images.to(device)
            if images.dtype == torch.uint8:
                images = prepare_cached_inputs(images, config["normalization"])
            with torch.autocast(
                device_type=device.type,
                enabled=device.type == "cuda" and config.get("precision") == "auto",
            ):
                features.append(model.encoder(images).float().cpu())
        labels.append(targets)
    return TensorDataset(torch.cat(features), torch.cat(labels))


@managed_run
def train_downstream(config: dict) -> None:
    validate_training_config(config, "downstream")
    seed_everything(int(config["seed"]))
    torch.set_num_threads(int(config.get("torch_num_threads", 4)))
    device = device_from_config(config.get("device", "auto"))
    amp = device.type == "cuda" and config["precision"] == "auto"
    config = dict(config)
    config["effective_precision"] = "float16" if amp else "float32"
    train_frame = read_manifest(config["train_manifest"])
    dev_frame = read_manifest(config["development_manifest"])
    reject_final_partition(config["train_manifest"], train_frame)
    reject_final_partition(config["development_manifest"], dev_frame)
    assert_patient_disjoint(train_frame, dev_frame)
    train_frame = select_patient_cohort(
        train_frame,
        config["sampled_patients"],
        float(config["label_fraction"]),
        int(config["seed"]),
    )
    mode = config["mode"]
    is_ssl = mode != "supervised"
    linear = mode in {"ssl_linear", "simclr_linear"}
    actual_method = config.get("ssl_method") if is_ssl else None
    config["identifier_normalization"] = {
        "train_restored_rows": train_frame.attrs.get("patient_id_padding_restored_rows", 0),
        "development_restored_rows": dev_frame.attrs.get("patient_id_padding_restored_rows", 0),
        "cohort_restored_ids": train_frame.attrs.get("cohort_patient_id_padding_restored", 0),
    }
    provenance = {
        key: file_hash(config[key])
        for key in ("train_manifest", "development_manifest", "sampled_patients")
    }
    if is_ssl:
        provenance["ssl_checkpoint"] = file_hash(config["ssl_checkpoint"])
    weights = positive_weights(train_frame, config.get("pos_weight_max"))
    if config.get("training_cache_manifest"):
        provenance["training_cache_manifest"] = file_hash(config["training_cache_manifest"])
    config["provenance"] = provenance
    config["positive_weights"] = weights.tolist()
    signature = config_signature(config)
    output_dir = Path(config["output_dir"])
    resume_path = output_dir / "last.pt"
    resumed = None
    if resume_path.exists():
        if not config["resume"]:
            raise ValueError("Output directory contains a checkpoint; choose a new output_dir")
        resumed = torch.load(resume_path, map_location="cpu", weights_only=False)
        if resumed.get("training_signature") != signature:
            raise ValueError(
                "Resume configuration, data cohort, or initialization checkpoint changed"
            )
    elif any((output_dir / name).exists() for name in ("best.pt", "metrics.json")):
        raise ValueError(
            "Existing run artifacts without a resumable checkpoint; choose a new output_dir"
        )
    if (output_dir / "selection_lock.json").exists():
        raise ValueError("A selected run is immutable; use a new output_dir for further training")

    model = MultiLabelClassifier(
        config["encoder"],
        len(TARGETS),
        freeze_encoder=linear,
        pretrained=config["imagenet_pretrained"] and resumed is None,
        weights_name=config.get("imagenet_weights"),
        standardize_features=linear and config.get("standardize_features", False),
    )
    if is_ssl:
        source = torch.load(config["ssl_checkpoint"], map_location="cpu", weights_only=False)
        if source.get("method") != actual_method:
            raise ValueError("SSL method does not match initialization checkpoint")
        source_config = source.get("config", {})
        for key in ("encoder", "image_size"):
            if source_config.get(key) != config[key]:
                raise ValueError(f"SSL checkpoint {key} does not match downstream comparison")
        source_normalization = source_config.get("augmentation", {}).get(
            "normalization", {"mean": 0.5, "std": 0.5}
        )
        if source_normalization != config.get("normalization", {"mean": 0.5, "std": 0.5}):
            raise ValueError("SSL checkpoint normalization does not match downstream preprocessing")
        model.load_pretrained_encoder(source)
    model.to(device)
    parameters = [p for p in model.parameters() if p.requires_grad]
    if not linear and config.get("encoder_learning_rate") is not None:
        parameters = [
            {"params": model.encoder.parameters(), "lr": float(config["encoder_learning_rate"])},
            {"params": model.classifier.parameters()},
        ]
    optimizer = AdamW(
        parameters, lr=float(config["learning_rate"]), weight_decay=float(config["weight_decay"])
    )
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    scheduler = (
        torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=int(config["epochs"]))
        if config.get("schedule") == "cosine" and not linear
        else None
    )
    criterion = nn.BCEWithLogitsLoss(pos_weight=weights.to(device))
    start_epoch, best_score, best_epoch, history = 1, -np.inf, 0, []
    if resumed is not None:
        model.load_state_dict(resumed["model"])
        optimizer.load_state_dict(resumed["optimizer"])
        scaler.load_state_dict(resumed["scaler"])
        if scheduler is not None:
            scheduler.load_state_dict(resumed["scheduler"])
        start_epoch = int(resumed["epoch"]) + 1
        best_score, best_epoch, history = (
            resumed["best_score"],
            resumed["best_epoch"],
            resumed["history"],
        )

    size, batch, workers = (
        int(config["image_size"]),
        int(config["batch_size"]),
        int(config["num_workers"]),
    )
    if linear:
        print("Extracting frozen training/development features once", flush=True)
        train_data = extract_features(
            model,
            train_frame,
            size,
            batch,
            workers,
            device,
            config["multiprocessing_context"],
            config,
        )
        dev_data = extract_features(
            model,
            dev_frame,
            size,
            batch,
            workers,
            device,
            config["multiprocessing_context"],
            config,
        )
        train_model = model.classifier
        if config.get("standardize_features") and resumed is None:
            features = train_data.tensors[0]
            with torch.no_grad():
                train_model.mean.copy_(features.mean(0))
                train_model.std.copy_(features.std(0).clamp_min(1e-6))
                if config.get("notebook_sampling"):
                    fresh_head = nn.Linear(features.shape[1], len(TARGETS)).to(device)
                    train_model.linear.load_state_dict(fresh_head.state_dict())
        loader_workers = 0
    elif config.get("cached_downstream"):
        train_data = CachedTargetDataset(
            train_frame,
            config["downstream_cache_path"],
            size,
            config.get("training_cache_manifest"),
        )
        dev_data = CachedTargetDataset(
            dev_frame, config["downstream_cache_path"], size, config.get("training_cache_manifest")
        )
        train_model = model
        loader_workers = workers
    else:
        train_data = CheXpertDataset(
            train_frame, supervised_transform(size, True, config["supervised_augmentation"])
        )
        dev_data = CheXpertDataset(dev_frame, supervised_transform(size, False))
        train_model = model
        loader_workers = workers
    shuffle_generator = torch.Generator()
    worker_generator = torch.Generator()
    sampler_options = {
        "batch_size": batch,
        "sampler": RandomSampler(train_data, generator=shuffle_generator),
    }
    batch_sampler = None
    if config.get("notebook_sampling"):
        batch_sampler = NotebookBatchSampler(
            len(train_data), batch, int(config["seed"]), "linear" if linear else "finetune"
        )
        sampler_options = {"batch_sampler": batch_sampler}
    train_loader = DataLoader(
        train_data,
        **sampler_options,
        generator=worker_generator,
        num_workers=loader_workers,
        multiprocessing_context=config["multiprocessing_context"] if loader_workers else None,
        pin_memory=device.type == "cuda",
    )
    dev_loader = DataLoader(
        dev_data,
        batch_size=batch,
        shuffle=False,
        num_workers=loader_workers,
        multiprocessing_context=config["multiprocessing_context"] if loader_workers else None,
        generator=torch.Generator().manual_seed(0),
    )
    if resumed is not None:
        restore_rng_state(resumed["rng_state"])
    save_run_metadata(output_dir, config)
    train_frame[["patient_id"]].drop_duplicates().to_csv(
        output_dir / "sampled_patients.csv", index=False
    )
    total_started = time.perf_counter()
    for epoch in range(start_epoch, int(config["epochs"]) + 1):
        if batch_sampler is not None:
            batch_sampler.epoch = epoch - 1
        shuffle_generator.manual_seed(int(config["seed"]) + epoch)
        worker_generator.manual_seed(int(config["seed"]) + epoch)
        if history and epoch - 1 - best_epoch >= int(config["early_stopping_patience"]):
            break
        started = time.perf_counter()
        train_model.train()
        if linear:
            model.encoder.eval()
        if device.type == "cuda":
            torch.cuda.reset_peak_memory_stats(device)
        loss_sum, records = 0.0, 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            if inputs.dtype == torch.uint8:
                inputs = augment_grayscale_batch(
                    inputs,
                    size,
                    {**config["supervised_augmentation"], "normalization": config["normalization"]},
                    channels_last=False,
                )
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=amp and not linear):
                loss = criterion(train_model(inputs), targets)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Nonfinite training loss at epoch {epoch}")
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.item() * len(inputs)
            records += len(inputs)
        if scheduler is not None:
            scheduler.step()
        truths, scores = predict(
            train_model,
            dev_loader,
            device,
            config.get("normalization"),
            amp=amp and not linear and config.get("cached_downstream", False),
        )
        metrics = multilabel_metrics(truths, scores)
        metrics.update(
            epoch=epoch,
            train_loss=loss_sum / records,
            seconds=time.perf_counter() - started,
            peak_gpu_memory_gb=torch.cuda.max_memory_allocated(device) / 2**30
            if device.type == "cuda"
            else 0.0,
        )
        history.append(metrics)
        score = metrics["macro_auroc"]
        print(
            f"epoch={epoch} train_loss={metrics['train_loss']:.4f} dev_auroc={score:.4f}",
            flush=True,
        )
        if np.isfinite(score) and score > best_score:
            best_score, best_epoch = score, epoch
            thresholds = select_thresholds(
                truths, scores, config.get("threshold_method", "balanced_accuracy")
            )
            atomic_torch_save(
                {
                    "model": model.state_dict(),
                    "config": config,
                    "epoch": epoch,
                    "training_signature": signature,
                    "thresholds": thresholds,
                },
                output_dir / "best.pt",
            )
            save_json(output_dir / "thresholds.json", thresholds)
        atomic_torch_save(
            {
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scaler": scaler.state_dict(),
                "scheduler": scheduler.state_dict() if scheduler else None,
                "config": config,
                "epoch": epoch,
                "best_score": best_score,
                "best_epoch": best_epoch,
                "history": history,
                "training_signature": signature,
                "rng_state": capture_rng_state(),
            },
            resume_path,
        )
    if not math.isfinite(best_score):
        raise ValueError("Development AUROC is undefined; no selectable checkpoint was produced")
    best_metrics = next(row for row in history if row["epoch"] == best_epoch)
    best = torch.load(output_dir / "best.pt", map_location=device, weights_only=False)
    model.load_state_dict(best["model"])
    truths, scores = predict(
        train_model,
        dev_loader,
        device,
        config.get("normalization"),
        amp=amp and not linear and config.get("cached_downstream", False),
    )

    best_metrics = {
        **best_metrics,
        **threshold_metrics(
            truths, scores, best["thresholds"], notebook=config.get("threshold_method") == "youden"
        ),
    }
    save_json(
        output_dir / "metrics.json",
        {
            **best_metrics,
            "partition": "development",
            "mode": mode,
            "ssl_method": actual_method,
            "label_fraction": config["label_fraction"],
            "seed": config["seed"],
            "history": history,
            "best_dev_auroc": best_score,
            "best_epoch": best_epoch,
            "positive_weights": weights.tolist(),
            "training_signature": signature,
        },
    )
    finish_run(
        output_dir,
        epochs_completed=history[-1]["epoch"],
        invocation_seconds=time.perf_counter() - total_started,
        training_seconds=sum(row["seconds"] for row in history),
        checkpoint_path=str(output_dir / "best.pt"),
        checkpoint_bytes=(output_dir / "best.pt").stat().st_size,
    )


def train_downstream_cli(argv=None) -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", type=Path, required=True)
    train_downstream(load_config(parser.parse_args(argv).config))

## Evaluate Checkpoint

Selection locks are verified before opening held-out records. Preprocessing and architecture come from the selected checkpoint.

In [99]:
"""Evaluate a development-locked image-only checkpoint on official validation."""


@managed_run
def evaluate(config: dict) -> None:
    config = dict(config)
    required = {
        "checkpoint",
        "selection_lock",
        "thresholds",
        "manifest",
        "output_dir",
        "batch_size",
        "seed",
        "bootstrap_samples",
    }
    if required.difference(config):
        raise ValueError(f"Missing evaluation keys: {sorted(required.difference(config))}")
    if int(config["batch_size"]) < 1 or int(config["bootstrap_samples"]) < 1:
        raise ValueError("Evaluation batch size and bootstrap sample count must be positive")
    output_dir = Path(config["output_dir"])
    if (output_dir / "metrics.json").exists():
        raise ValueError("Evaluation results already exist; reuse them instead of reevaluating")
    checkpoint = torch.load(config["checkpoint"], map_location="cpu", weights_only=False)
    training, lock = verify_selection(config, checkpoint)
    config["training_config"] = training
    config["source_run_id"] = lock["source_run_id"]
    for key in ("encoder", "image_size", "target_policy", "supervised_augmentation"):
        config[key] = training[key]
    train = read_manifest(training["train_manifest"])
    dev = read_manifest(training["development_manifest"])
    for key in ("train_manifest", "development_manifest"):
        if file_hash(training[key]) != training["provenance"][key]:
            raise ValueError("Training/development manifest changed since selection")
    frame = read_manifest(config["manifest"])
    assert_patient_disjoint(train, dev, frame)
    config["provenance"] = {
        key: file_hash(config[key])
        for key in ("checkpoint", "selection_lock", "thresholds", "manifest")
    }
    if int(config.get("num_workers", 0)) != 0:
        raise ValueError("Notebook evaluation requires num_workers=0")
    device = device_from_config(config.get("device", "auto"))
    # All weights come from the selected checkpoint; evaluation never downloads initialization weights.
    model = MultiLabelClassifier(
        training["encoder"],
        len(TARGETS),
        pretrained=False,
        standardize_features=training.get("standardize_features", False)
        and training["mode"] in {"ssl_linear", "simclr_linear"},
    )
    model.load_state_dict(checkpoint["model"], strict=True)
    model.to(device).eval()
    loader = DataLoader(
        CheXpertDataset(
            frame,
            supervised_transform(
                training["image_size"], False, training.get("supervised_augmentation")
            ),
        ),
        batch_size=int(config["batch_size"]),
        shuffle=False,
        num_workers=int(config.get("num_workers", 0)),
        multiprocessing_context=training["multiprocessing_context"]
        if int(config.get("num_workers", 0))
        else None,
    )
    save_run_metadata(output_dir, config)
    probabilities = []
    with torch.inference_mode():
        for images, _ in loader:
            with torch.autocast(
                device_type=device.type,
                enabled=device.type == "cuda"
                and training.get("cached_downstream")
                and training["precision"] == "auto",
            ):
                logits = model(images.to(device))
            probabilities.append(torch.sigmoid(logits.float()).cpu().numpy())
    scores = np.concatenate(probabilities)
    if not np.isfinite(scores).all():
        raise FloatingPointError("Nonfinite evaluation probabilities")
    targets = frame[target_columns()].to_numpy(dtype=np.float32)
    thresholds = load_config(Path(config["thresholds"]))
    metrics = multilabel_metrics(targets, scores)
    metrics.update(
        threshold_metrics(
            targets, scores, thresholds, notebook=training.get("threshold_method") == "youden"
        )
    )
    if config.get("per_label_bootstrap"):
        intervals = patient_bootstrap_auroc(
            targets,
            scores,
            frame.patient_id.to_numpy(),
            int(config["seed"]),
            int(config["bootstrap_samples"]),
        )
        metrics["macro_auroc_ci95"] = intervals["macro_auroc"]
        metrics["per_label_auroc_ci95"] = intervals["per_label"]
    else:
        metrics["macro_auroc_ci95"] = patient_bootstrap_macro_auroc(
            targets,
            scores,
            frame.patient_id.to_numpy(),
            int(config["seed"]),
            int(config["bootstrap_samples"]),
        )
    metrics.update(
        partition="final_validation",
        source_run_id=lock["source_run_id"],
        mode=training["mode"],
        ssl_method=training.get("ssl_method"),
        label_fraction=training["label_fraction"],
        seed=training["seed"],
        training_signature=checkpoint["training_signature"],
    )
    columns = [
        "image_path",
        "patient_id",
        "study_id",
        "Frontal/Lateral",
        "AP/PA",
        "Age",
        "Sex",
        *target_columns(),
    ]
    predictions = frame[[column for column in columns if column in frame]].copy()
    for index, label in enumerate(TARGETS):
        predictions[f"probability_{label}"] = scores[:, index]
    predictions.to_csv(output_dir / "predictions.csv", index=False)
    save_json(output_dir / "metrics.json", metrics)
    finish_run(
        output_dir, evaluated_images=len(frame), evaluated_patients=frame.patient_id.nunique()
    )


def evaluate_checkpoint_cli(argv=None) -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", type=Path, required=True)
    evaluate(load_config(parser.parse_args(argv).config))

## Aggregate Results

In [100]:
"""Aggregate metric artifacts with explicit partition and initialization identity."""


def collect_results(root: Path, partition: str) -> pd.DataFrame:
    rows = []
    for path in sorted(root.rglob("metrics.json")):
        if any(part in {"smoke", "archive"} for part in path.relative_to(root).parts):
            continue
        payload = load_config(path)
        if payload.get("partition") != partition:
            continue
        metadata_path = path.parent / "run_metadata.json"
        if not metadata_path.exists():
            continue
        metadata = load_config(metadata_path)
        if metadata.get("status") != "complete":
            continue
        config = metadata["config"]
        training = config.get("training_config", config)
        mode = payload.get("mode", training.get("mode"))
        ssl_method = payload.get("ssl_method", training.get("ssl_method"))
        initialization = ssl_method or (
            "imagenet" if training.get("imagenet_pretrained") else "scratch"
        )
        method = f"{initialization}_{mode}"
        modality = training.get(
            "pretraining_modality",
            "image-only SSL"
            if ssl_method
            else "ImageNet"
            if training.get("imagenet_pretrained")
            else "none",
        )
        metrics = {
            key: value
            for key, value in payload.items()
            if key.startswith(
                (
                    "macro_",
                    "auroc_",
                    "auprc_",
                    "sensitivity_",
                    "specificity_",
                    "f1_",
                    "balanced_accuracy_",
                )
            )
            and isinstance(value, (float, int))
        }
        rows.append(
            {
                "run_id": payload.get("source_run_id", path.parent.name),
                "partition": partition,
                "method": method,
                "protocol": mode,
                "architecture": training.get("encoder"),
                "label_fraction": payload.get("label_fraction", training.get("label_fraction")),
                "seed": payload.get("seed", training.get("seed")),
                "pretraining_modality": modality,
                "metrics_path": str(path),
                **metrics,
            }
        )
    raw = pd.DataFrame(rows)
    if not raw.empty:
        identity = ["architecture", "method", "pretraining_modality", "label_fraction", "seed"]
        if raw.duplicated(identity, keep=False).any():
            raise ValueError(
                "Duplicate seed/method/budget results; select a single experiment root"
            )
    return raw


def aggregate_results_cli(argv=None) -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--root", type=Path, default=Path("outputs"))
    parser.add_argument("--output-dir", type=Path, default=Path("outputs/reports"))
    parser.add_argument(
        "--plots", action="store_true", help="Write notebook label-efficiency table and curve"
    )
    parser.add_argument(
        "--partition", choices=("development", "final_validation"), default="final_validation"
    )
    args = parser.parse_args(argv)
    raw = collect_results(args.root, args.partition)
    output = args.output_dir / args.partition
    output.mkdir(parents=True, exist_ok=True)
    raw.to_csv(output / "raw_metrics.csv", index=False)
    if raw.empty:
        print(f"No completed {args.partition} metric artifacts found")
        return
    keys = ["partition", "architecture", "method", "pretraining_modality", "label_fraction"]
    metrics = [
        column
        for column in raw
        if column.startswith(
            (
                "macro_",
                "auroc_",
                "auprc_",
                "sensitivity_",
                "specificity_",
                "f1_",
                "balanced_accuracy_",
            )
        )
    ]
    summary = raw.groupby(keys, dropna=False)[metrics].agg(["mean", "std", "count"])
    summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
    summary.reset_index().to_csv(output / "label_efficiency_summary.csv", index=False)
    if args.plots:
        write_label_efficiency_plot(raw, output)


def write_label_efficiency_plot(raw: pd.DataFrame, output: Path) -> None:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    grouped = (
        raw.groupby(["method", "label_fraction"])
        .macro_auroc.agg(["mean", "std", "count"])
        .reset_index()
    )
    grouped["cell"] = grouped.apply(
        lambda r: (
            f"{r['mean']:.3f} ± {0 if pd.isna(r['std']) else r['std']:.3f} (n={int(r['count'])})"
        ),
        axis=1,
    )
    grouped.pivot(index="method", columns="label_fraction", values="cell").to_csv(
        output / "macro_auroc_table.csv"
    )
    figure, axis = plt.subplots(figsize=(8, 5))
    for method, sub in grouped.groupby("method"):
        axis.errorbar(
            sub.label_fraction * 100,
            sub["mean"],
            yerr=sub["std"].fillna(0),
            marker="o",
            capsize=3,
            label=method,
        )
    axis.set_xscale("log")
    axis.set_xlabel("Labelled patients (%)")
    axis.set_ylabel("Macro AUROC")
    axis.set_title(f"Label efficiency ({raw.partition.iloc[0]})")
    axis.legend(fontsize=7, ncol=2)
    axis.grid(alpha=0.3)
    figure.tight_layout()
    figure.savefig(output / "label_efficiency.png", dpi=130)
    plt.close(figure)

## Analyze Errors

In [101]:
"""Create an auditable error-analysis worksheet from prediction artifacts."""


def analyze_errors_cli(argv=None) -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--predictions", type=Path, required=True)
    parser.add_argument("--thresholds", type=Path, required=True)
    parser.add_argument("--run-id", required=True)
    parser.add_argument("--output", type=Path, required=True)
    args = parser.parse_args(argv)
    frame = pd.read_csv(args.predictions)
    thresholds = json.loads(args.thresholds.read_text(encoding="utf-8"))
    rows = []
    for _, record in frame.iterrows():
        for label in TARGETS:
            probability = record[f"probability_{label}"]
            truth = record[f"target_{label}"]
            predicted = int(probability >= float(thresholds[label]))
            if predicted != truth:
                rows.append(
                    {
                        "run_id": args.run_id,
                        "image_id": record["image_path"],
                        "patient_id": record["patient_id"],
                        "study_id": record["study_id"],
                        "view": record["Frontal/Lateral"],
                        "label": label,
                        "true_label": truth,
                        "predicted_probability": probability,
                        "selected_threshold": float(thresholds[label]),
                        "attribution_artifact_path": "",
                        "reviewer_observation": "",
                    }
                )
    args.output.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(args.output, index=False)

## Lock Selection

In [102]:
"""Lock a completed run after reviewing its development results."""


def lock_selection_cli(argv=None) -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--run-dir", type=Path, required=True)
    parser.add_argument(
        "--reason", required=True, help="Decision based exclusively on development results"
    )
    args = parser.parse_args(argv)
    print(lock_run(args.run_dir, args.reason))

## Train Vlm

In [103]:
"""Train a frozen VLM feature probe or permitted VLM image-encoder fine-tune."""


class Records(Dataset):
    def __init__(self, frame: pd.DataFrame) -> None:
        self.frame = frame.reset_index(drop=True)

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int):
        row = self.frame.iloc[index]
        with Image.open(row.image_path) as source:
            image = source.convert("RGB")
        return image, torch.tensor(row[target_columns()].to_numpy(dtype=np.float32))


def train_vlm_cli(argv=None) -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", type=Path, required=True)
    args = parser.parse_args(argv)
    config = load_config(args.config)
    validate_vlm_config(config)
    mode = config["mode"]
    if mode not in {"vlm_linear", "vlm_finetune"}:
        raise ValueError("mode must be vlm_linear or vlm_finetune")
    seed_everything(int(config["seed"]))
    device = device_from_config(config.get("device", "auto"))
    output_dir = Path(config["output_dir"])
    output_dir.mkdir(parents=True, exist_ok=True)
    if output_dir.exists() and any(output_dir.iterdir()):
        raise ValueError("VLM runs are experimental and not resumable; use a new output_dir")
    train = read_manifest(config["train_manifest"])
    dev = read_manifest(config["development_manifest"])
    reject_final_partition(config["train_manifest"], train)
    reject_final_partition(config["development_manifest"], dev)
    assert_patient_disjoint(train, dev)
    if not config.get("sampled_patients"):
        raise ValueError("VLM training requires the same persisted patient cohort as other methods")
    train = select_patient_cohort(
        train, config["sampled_patients"], config["label_fraction"], config["seed"]
    )
    config["implementation_sha256"] = implementation_hash()
    config["provenance"] = {
        key: file_hash(config[key])
        for key in ("train_manifest", "development_manifest", "sampled_patients")
    }
    save_run_metadata(output_dir, config)
    encoder = HuggingFaceVLM(config["checkpoint"], config.get("revision"))
    model = VLMClassifier(encoder, len(TARGETS), freeze_encoder=mode == "vlm_linear").to(device)

    def collate(rows):
        images, targets = zip(*rows)
        pixels = encoder.processor(images=list(images), return_tensors="pt")["pixel_values"]
        return pixels, torch.stack(list(targets))

    train_loader = DataLoader(
        Records(train), batch_size=int(config["batch_size"]), shuffle=True, collate_fn=collate
    )
    dev_loader = DataLoader(
        Records(dev), batch_size=int(config["batch_size"]), shuffle=False, collate_fn=collate
    )
    values = train[target_columns()].to_numpy(dtype=np.float32)
    pos_weight = torch.tensor(
        (len(values) - values.sum(0)) / np.clip(values.sum(0), 1, None), device=device
    )
    if mode == "vlm_finetune":
        optimizer = AdamW(
            [
                {
                    "params": model.encoder.parameters(),
                    "lr": float(config.get("encoder_learning_rate", config["learning_rate"])),
                },
                {"params": model.classifier.parameters(), "lr": float(config["learning_rate"])},
            ],
            weight_decay=float(config["weight_decay"]),
        )
    else:
        optimizer = AdamW(
            model.classifier.parameters(),
            lr=float(config["learning_rate"]),
            weight_decay=float(config["weight_decay"]),
        )
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    best, best_epoch, history = -np.inf, 0, []
    for epoch in range(1, int(config["epochs"]) + 1):
        model.train()
        for pixels, targets in train_loader:
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(pixels.to(device)), targets.to(device))
            if not torch.isfinite(loss):
                raise FloatingPointError("Nonfinite VLM training loss")
            loss.backward()
            optimizer.step()
        model.eval()
        scores, truths = [], []
        with torch.inference_mode():
            for pixels, targets in dev_loader:
                scores.append(torch.sigmoid(model(pixels.to(device))).cpu().numpy())
                truths.append(targets.numpy())
        truth, score = np.concatenate(truths), np.concatenate(scores)
        metrics = multilabel_metrics(truth, score)
        metrics["epoch"] = epoch
        history.append(metrics)
        if np.isfinite(metrics["macro_auroc"]) and metrics["macro_auroc"] > best:
            best, best_epoch = metrics["macro_auroc"], epoch
            atomic_torch_save(
                {"model": model.state_dict(), "config": config, "epoch": epoch},
                output_dir / "best.pt",
            )
            save_json(output_dir / "thresholds.json", select_thresholds(truth, score))
        if epoch - best_epoch >= int(config.get("early_stopping_patience", 10)):
            break
    save_json(
        output_dir / "metrics.json",
        {
            "mode": mode,
            "label_fraction": config["label_fraction"],
            "best_dev_auroc": best,
            "best_epoch": best_epoch,
            "history": history,
            "positive_weights": pos_weight.cpu().tolist(),
        },
    )
    train[["patient_id"]].drop_duplicates().to_csv(output_dir / "sampled_patients.csv", index=False)

    finish_run(output_dir, best_epoch=best_epoch, pretraining_modality="image-text VLM")

## Evaluate Vlm Zeroshot

In [104]:
"""Run a versioned, label-free VLM prompt baseline on a prepared manifest."""


def evaluate_vlm_zeroshot_cli(argv=None) -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", type=Path, required=True)
    args = parser.parse_args(argv)
    config = load_config(args.config)
    validate_vlm_config(config)
    frame = read_manifest(config["manifest"])
    reject_final_partition(config["manifest"], frame)
    if not frame["image_path"].size:
        raise ValueError("VLM development manifest is empty")
    device = device_from_config(config.get("device", "auto"))
    output_dir = Path(config["output_dir"])
    output_dir.mkdir(parents=True, exist_ok=True)
    if output_dir.exists() and any(output_dir.iterdir()):
        raise ValueError("Use a new output directory for VLM development evaluation")
    config["implementation_sha256"] = implementation_hash()
    config["provenance"] = {"manifest": file_hash(config["manifest"])}
    config["resolved_prompts"] = load_config(config["prompt_set"])["prompts"]
    save_run_metadata(output_dir, config)
    prompts = load_config(config["prompt_set"])["prompts"]
    model = HuggingFaceVLM(config["checkpoint"], config.get("revision")).to(device).eval()
    concepts = aggregate_prompt_embeddings(model, prompts, device)
    all_scores: list[np.ndarray] = []
    batch_size = int(config.get("batch_size", 32))
    with torch.inference_mode():
        for start in range(0, len(frame), batch_size):
            paths = frame.iloc[start : start + batch_size]["image_path"].tolist()
            images = [Image.open(path).convert("RGB") for path in paths]
            batch = model.processor(images=images, return_tensors="pt")
            for image in images:
                image.close()
            embeddings = model.image_features(batch["pixel_values"].to(device))
            all_scores.append(zero_shot_scores(embeddings, concepts))
    scores = np.concatenate(all_scores)
    targets = frame[target_columns()].to_numpy(dtype=np.float32)
    save_json(
        output_dir / "metrics.json",
        {
            **multilabel_metrics(targets, scores),
            "partition": "development",
            "pretraining_modality": "image-text VLM",
            "calibrated": False,
            "prompt_set": config["prompt_set"],
        },
    )
    output = frame[
        ["image_path", "patient_id", "study_id", "Frontal/Lateral", *target_columns()]
    ].copy()
    for index, label in enumerate(TARGETS):
        output[f"probability_{label}"] = scores[:, index]
    output.to_csv(output_dir / "predictions.csv", index=False)

    finish_run(output_dir, evaluated_images=len(frame))

## Run Experiments

The staged runner validates plans and artifact reuse. Calls are dispatched to notebook functions; no deleted script files or generated source tree are needed.

In [105]:
"""Materialize and execute the five-method image-only matrix in explicit stages."""


def smoke_settings(matrix: dict) -> tuple[dict, dict, Path, str]:
    """Version smoke outputs by their settings, source inputs, and implementation."""
    if matrix.get("notebook_smoke"):
        suite, root, source, _ = notebook_smoke_suite(matrix)
        return suite["pretrain-simclr"], suite["downstream-linear"], root, source["manifest"]
    pretrain = load_config(Path("configs/pretrain/simclr_smoke.yaml"))
    downstream = load_config(Path("configs/downstream/smoke.yaml"))
    full_pretrain = load_config(Path(matrix["pretrain_configs"]["simclr"]))
    source_config = Path(matrix["downstream_templates"]["ssl_linear"])
    source = load_config(source_config)
    pretrain.update(
        development_manifest=full_pretrain["development_manifest"],
        augmentation=full_pretrain["augmentation"],
        seed=full_pretrain["seed"],
        resume=True,
    )
    downstream.update(
        seed=source["seed"],
        encoder=pretrain["encoder"],
        image_size=pretrain["image_size"],
        resume=True,
    )
    inputs = {
        path: file_hash(path)
        for path in {
            full_pretrain["manifest"],
            full_pretrain["development_manifest"],
            source["train_manifest"],
            source["development_manifest"],
            source["sampled_patients"],
        }
    }
    fingerprint = config_signature(
        {
            "pretrain": pretrain,
            "downstream": downstream,
            "inputs": inputs,
            "pretrain_implementation": implementation_hash(Path("train_pretrain")),
            "downstream_implementation": implementation_hash(Path("train_downstream")),
        }
    )[:16]
    root = Path(pretrain["output_dir"]).parent / "runs" / fingerprint
    pretrain["output_dir"] = str(root / "pretrain")
    downstream.update(
        output_dir=str(root / "downstream"), ssl_checkpoint=str(root / "pretrain" / "best.pt")
    )
    return pretrain, downstream, root, full_pretrain["manifest"]


def notebook_smoke_suite(matrix):
    """Bounded checks of every SSL objective and all downstream initialization paths."""
    plan = build_plan(matrix)
    pretrain = plan["pretrain"][0]
    source = next(
        c
        for c in plan["downstream"]
        if c.get("ssl_method") == "simclr"
        and c["mode"] == "ssl_linear"
        and c["label_fraction"] == matrix.get("smoke_budget", 0.05)
    )
    inputs = {
        path: file_hash(path)
        for path in {
            pretrain["manifest"],
            pretrain["development_manifest"],
            source["train_manifest"],
            source["development_manifest"],
            source["sampled_patients"],
            pretrain["cache_source_manifest"],
        }
    }
    fingerprint = config_signature(
        {
            "matrix": matrix,
            "plan": plan,
            "inputs": inputs,
            "pretrain_implementation": implementation_hash(Path("train_pretrain")),
            "downstream_implementation": implementation_hash(Path("train_downstream")),
        }
    )[:16]
    root = Path(matrix["pretrain_output_root"]).parent / "smoke" / fingerprint
    data_root = Path(matrix["splits_dir"]).parent / "smoke" / fingerprint
    suite = {}
    for config in plan["pretrain"]:
        name = f"pretrain-{config['method']}"
        suite[name] = {
            **config,
            "output_dir": str(root / name),
            "epochs": 2,
            "batch_size": 64,
            "max_steps_per_epoch": 4,
            "manifest": str(data_root / "pretrain.csv"),
            "cache_path": str(data_root / "pretrain_gray.npy"),
            "resume": True,
        }
        suite[name].pop("cache_source_manifest", None)
    for name, method, mode in [
        ("linear", "simclr", "ssl_linear"),
        ("scratch", None, "supervised"),
        ("moco", "moco", "ssl_finetune"),
        ("imagenet", None, "supervised"),
    ]:
        config = next(
            c
            for c in plan["downstream"]
            if c["mode"] == mode
            and c.get("ssl_method") == method
            and c["imagenet_pretrained"] == (name == "imagenet")
        )
        config = {
            **config,
            "output_dir": str(root / f"downstream-{name}"),
            "epochs": 2,
            "batch_size": 64 if mode != "ssl_linear" else 512,
            "early_stopping_patience": 2,
            "label_fraction": 1.0,
            "resume": True,
            "train_manifest": str(data_root / "downstream_train.csv"),
            "development_manifest": str(data_root / "development.csv"),
            "sampled_patients": str(data_root / "cohort.json"),
            "training_cache_manifest": str(data_root / "all_training.csv"),
            "downstream_cache_path": str(data_root / "downstream_gray.npy"),
        }
        if method:
            config["ssl_checkpoint"] = str(root / f"pretrain-{method}" / "last.pt")
        suite[f"downstream-{name}"] = config
    return suite, root, pretrain, source


def run_notebook_smoke(matrix):
    suite, root, pretrain_source, downstream_source = notebook_smoke_suite(matrix)
    for name, config in suite.items():
        path = root / f"{name}.json"
        if path.exists() and load_config(path) != config:
            raise ValueError(f"Smoke configuration artifact changed: {path}")
        save_json(path, config)
    source_path = root / "source-downstream.json"
    save_json(source_path, downstream_source)
    data_root = Path(suite["downstream-linear"]["train_manifest"]).parent
    for command in [
        ["test"],
        [
            "prepare_smoke_manifest",
            "--source",
            pretrain_source["manifest"],
            "--output",
            suite["pretrain-simclr"]["manifest"],
        ],
        [
            "prepare_downstream_smoke",
            "--config",
            str(source_path),
            "--output-dir",
            str(data_root),
        ],
    ]:
        execute_task(command)
    save_json(
        root / "tests_passed.json", {"implementation_sha256": implementation_hash(), "passed": True}
    )
    for name, config in suite.items():
        script = "train_pretrain.py" if name.startswith("pretrain-") else "train_downstream.py"
        metadata = Path(config["output_dir"]) / "run_metadata.json"
        if metadata.exists() and load_config(metadata).get("status") == "complete":
            previous = load_config(metadata)["config"]
            if not all(previous.get(k) == v for k, v in config.items()):
                raise ValueError(f"Completed smoke configuration changed: {metadata}")
            for key, digest in previous.get("provenance", {}).items():
                if key in previous and file_hash(previous[key]) != digest:
                    raise ValueError(f"Completed smoke inputs changed: {metadata}")
            if not all(
                (Path(config["output_dir"]) / artifact).is_file()
                for artifact in ("best.pt", "last.pt", "metrics.json")
            ):
                raise ValueError(f"Completed smoke artifacts missing: {metadata}")
            print(f"Reusing completed smoke: {name}", flush=True)
            continue
        execute_task(
            [script.removesuffix(".py"), "--config", str(root / f"{name}.json")],
        )
    print(
        f"Inspect {suite['pretrain-simclr']['output_dir']}/augmentation_pairs.png before full pretraining."
    )


def run_experiments_cli(argv=None) -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--config", type=Path, default=Path("configs/experiments/notebook_reference.yaml")
    )
    parser.add_argument(
        "--stage",
        choices=("plan", "prepare", "smoke", "pretrain", "downstream", "evaluate", "aggregate"),
        default="plan",
    )
    parser.add_argument(
        "--methods", nargs="+", help="Restrict SSL methods or scratch/imagenet arms"
    )
    parser.add_argument("--budgets", type=float, nargs="+")
    parser.add_argument("--seeds", type=int, nargs="+")
    parser.add_argument(
        "--partition",
        choices=("development", "final_validation"),
        default="final_validation",
        help="Partition for aggregate stage",
    )
    parser.add_argument(
        "--augmentation-reviewed",
        action="store_true",
        help="Confirm anatomical review of the completed smoke augmentation preview",
    )
    args = parser.parse_args(argv)
    matrix = load_config(args.config)
    if args.stage == "prepare":
        execute_task(
            ["prepare_data", "--config", matrix["data_config"]],
        )
    elif args.stage == "smoke":
        if matrix.get("notebook_smoke"):
            run_notebook_smoke(matrix)
            return
        pretrain, downstream, smoke_root, source_manifest = smoke_settings(matrix)
        for name, config in (("pretrain", pretrain), ("downstream", downstream)):
            path = smoke_root / f"{name}.json"
            if path.exists() and load_config(path) != config:
                raise ValueError(f"Smoke configuration artifact changed: {path}")
            if not path.exists():
                save_json(path, config)
        for command in (
            ["test"],
            ["prepare_smoke_manifest", "--source", source_manifest],
            [
                "train_pretrain",
                "--config",
                str(smoke_root / "pretrain.json"),
            ],
            [
                "prepare_downstream_smoke",
                "--config",
                matrix["downstream_templates"]["ssl_linear"],
            ],
            [
                "train_downstream",
                "--config",
                str(smoke_root / "downstream.json"),
            ],
        ):
            execute_task(command)
        save_json(
            smoke_root / "tests_passed.json",
            {"implementation_sha256": implementation_hash(), "passed": True},
        )
        print(f"Inspect {pretrain['output_dir']}/augmentation_pairs.png before full pretraining.")
    elif args.stage == "aggregate":
        command = ["aggregate_results", "--partition", args.partition]
        if matrix.get("notebook_smoke"):
            command += [
                "--root",
                str(Path(matrix["pretrain_output_root"]).parent),
                "--output-dir",
                str(Path(matrix["pretrain_output_root"]).parent / "reports"),
                "--plots",
            ]
        execute_task(
            command,
        )
    if args.stage in {"prepare", "smoke", "aggregate"}:
        return
    plan = build_plan(matrix)
    available = set(matrix["pretrain_configs"]) | {"scratch", "imagenet"}
    if args.methods and not set(args.methods).issubset(available):
        parser.error("Unknown method filter")
    if args.budgets and not set(args.budgets).issubset(set(matrix["budgets"])):
        parser.error("Unknown budget filter")
    if args.seeds and not set(args.seeds).issubset(set(matrix["seeds"])):
        parser.error("Unknown seed filter")
    root = Path(matrix["generated_dir"])
    commands = {}
    for stage, configs in plan.items():
        commands[stage] = []
        for config in configs:
            run_id = Path(config["output_dir"]).name
            path = root / stage / f"{run_id}.json"
            if path.exists() and load_config(path) != config:
                raise ValueError(f"Generated plan changed; use a new generated_dir: {path}")
            if not path.exists():
                save_json(path, config)
            script = "evaluate_checkpoint" if stage == "evaluate" else f"train_{stage}"
            commands[stage].append([script, "--config", str(path)])
    save_json(
        root / "plan.json",
        {"matrix": matrix, "counts": {k: len(v) for k, v in plan.items()}, "commands": commands},
    )
    print(
        f"Plan: {len(plan['pretrain'])} pretraining runs, {len(plan['downstream'])} downstream runs; {root}",
        flush=True,
    )
    if args.stage == "plan":
        return
    if args.stage in {"pretrain", "downstream", "evaluate"}:
        if args.stage == "pretrain":
            smoke, downstream_smoke, smoke_root, _ = smoke_settings(matrix)
            test_record = smoke_root / "tests_passed.json"
            if not test_record.exists() or load_config(test_record) != {
                "implementation_sha256": implementation_hash(),
                "passed": True,
            }:
                raise ValueError("Complete the test/smoke stage for this notebook implementation")
            settings = (
                list(notebook_smoke_suite(matrix)[0].values())
                if matrix.get("notebook_smoke")
                else [smoke, downstream_smoke]
            )
            for setting in settings:
                metadata = Path(setting["output_dir"]) / "run_metadata.json"
                if not metadata.exists() or load_config(metadata).get("status") != "complete":
                    raise ValueError("Complete the test/smoke stage before full pretraining")
                previous = load_config(metadata)["config"]
                entrypoint = "train_downstream.py" if "mode" in setting else "train_pretrain.py"
                if previous.get("implementation_sha256") != implementation_hash(Path(entrypoint)):
                    raise ValueError("Implementation changed; complete a new smoke run")
                if not all(previous.get(key) == value for key, value in setting.items()):
                    raise ValueError("Smoke configuration changed; complete a new smoke run")
                for key, digest in previous.get("provenance", {}).items():
                    if key in previous and file_hash(previous[key]) != digest:
                        raise ValueError("Smoke inputs changed; complete a new smoke run")
            if not args.augmentation_reviewed:
                raise ValueError(
                    "Inspect the smoke augmentation pairs, then pass --augmentation-reviewed"
                )
            preview = Path(smoke["output_dir"]) / "augmentation_pairs.png"
            save_json(
                root / "augmentation_review.json",
                {"preview": str(preview), "sha256": file_hash(preview)},
            )
        jobs = []
        for config, command in zip(plan[args.stage], commands[args.stage]):
            source = config
            if args.stage == "evaluate":
                source = next(
                    c
                    for c in plan["downstream"]
                    if str(Path(c["output_dir"]) / "best.pt") == config["checkpoint"]
                )
            method = (
                source.get("ssl_method")
                or source.get("method")
                or ("imagenet" if source.get("imagenet_pretrained") else "scratch")
            )
            if args.methods and method not in args.methods:
                continue
            if args.stage != "pretrain":
                if args.budgets and source["label_fraction"] not in args.budgets:
                    continue
                if args.seeds and source["seed"] not in args.seeds:
                    continue
            jobs.append((config, command))
        # Check every requested selection lock before any official validation is opened.
        if args.stage == "evaluate":
            import torch

            for config, _ in jobs:
                checkpoint = torch.load(
                    config["checkpoint"], map_location="cpu", weights_only=False
                )
                verify_selection(config, checkpoint)
        for config, command in jobs:
            metadata = Path(config["output_dir"]) / "run_metadata.json"
            if metadata.exists() and load_config(metadata).get("status") == "complete":
                previous = load_config(metadata)["config"]
                if previous.get("implementation_sha256") != implementation_hash(Path(command[0])):
                    raise ValueError(f"Completed run implementation changed: {metadata}")
                if not all(previous.get(key) == value for key, value in config.items()):
                    raise ValueError(f"Completed run differs from plan: {metadata}")
                for key, digest in previous.get("provenance", {}).items():
                    if key in previous and file_hash(previous[key]) != digest:
                        raise ValueError(f"Completed run input changed: {previous[key]}")
                for name in (
                    "metrics.json",
                    "best.pt" if args.stage != "evaluate" else "predictions.csv",
                    *(
                        ["last.pt"]
                        if args.stage == "pretrain"
                        and matrix.get("transfer_checkpoint") == "last.pt"
                        else []
                    ),
                ):
                    if not (Path(config["output_dir"]) / name).exists():
                        raise ValueError(f"Completed run is missing {name}: {metadata}")
                print(f"Reusing completed run: {config['output_dir']}", flush=True)
                continue
            execute_task(command)

In [106]:
TASKS = {
    "prepare_data": prepare_data_cli,
    "repair_pretrain_manifest": repair_pretrain_manifest_cli,
    "prepare_smoke_manifest": prepare_smoke_manifest_cli,
    "prepare_downstream_smoke": prepare_downstream_smoke_cli,
    "train_pretrain": train_pretrain_cli,
    "train_downstream": train_downstream_cli,
    "evaluate_checkpoint": evaluate_checkpoint_cli,
    "aggregate_results": aggregate_results_cli,
    "analyze_errors": analyze_errors_cli,
    "lock_selection": lock_selection_cli,
    "train_vlm": train_vlm_cli,
    "evaluate_vlm_zeroshot": evaluate_vlm_zeroshot_cli,
}


def execute_task(command):
    """Run a named notebook task, propagating errors and restoring the working directory."""
    if not command:
        raise ValueError("A task name is required")
    global ACTIVE_TASK_COMMAND
    previous_command = globals().get("ACTIVE_TASK_COMMAND")
    ACTIVE_TASK_COMMAND = [str(value) for value in command]
    try:
        with project_directory():
            if command[0] == "test":
                return subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
            if command[0] not in TASKS:
                raise ValueError(f"Unknown notebook task: {command[0]}")
            return TASKS[command[0]]([str(value) for value in command[1:]])
    finally:
        ACTIVE_TASK_COMMAND = previous_command


def run_stage(
    stage="plan",
    *,
    config="configs/experiments/notebook_reference.yaml",
    methods=None,
    budgets=None,
    seeds=None,
    partition="development",
    augmentation_reviewed=False,
):
    """Execute one explicit stage; planning never starts training."""
    argv = ["--stage", stage, "--config", str(config), "--partition", partition]
    for name, values in (("methods", methods), ("budgets", budgets), ("seeds", seeds)):
        if values is not None:
            if not values:
                raise ValueError(f"{name} must be nonempty when supplied")
            argv.extend([f"--{name}", *map(str, values)])
    if augmentation_reviewed:
        argv.append("--augmentation-reviewed")
    with project_directory():
        return run_experiments_cli(argv)


globals().pop("LOADED_IMPLEMENTATION_HASH", None)
LOADED_IMPLEMENTATION_HASH = implementation_hash()

## Workflow controls

Definitions above are the entire implementation. Enable and run **one stage at a time**, in this order: prepare → smoke → inspect preview → pretrain → downstream → development report → lock → evaluate → final report. Full stages remain disabled on Run All.

Preparation reads official validation only for manifest construction, image-quality and patient-disjointness checks; it does not display or use validation label prevalence for selection. Model selection and threshold fitting use development data exclusively.

Existing prepared data is intentionally protected from overwrite. If preparation already exists, reuse it. For a new protocol, choose separate data, output and plan paths in YAML. Never regenerate cohorts inside an ongoing comparison.

### Plan the reference matrix

In [117]:
RUN_PLAN = True
if RUN_PLAN:
    run_stage("plan")

Plan: 5 pretraining runs, 180 downstream runs; outputs/plans/notebook-consolidated


### Prepare data once

In [ ]:
RUN_PREPARE = True
if RUN_PREPARE:
    run_stage("prepare")

### Tests and bounded smoke

In [ ]:
RUN_SMOKE = False
if RUN_SMOKE:
    run_stage("smoke")
    from IPython.display import Image as PreviewImage
    from IPython.display import display

    smoke_config, _, _, _ = smoke_settings(
        load_config("configs/experiments/notebook_reference.yaml")
    )
    display(
        PreviewImage(
            filename=str(PROJECT_ROOT / smoke_config["output_dir"] / "augmentation_pairs.png")
        )
    )

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/notebook-reference/manifests/development.csv'

### Full pretraining after anatomical review

In [110]:
AUGMENTATION_REVIEWED = False  # Set only after inspecting the matching smoke preview.
RUN_PRETRAIN = False
if RUN_PRETRAIN:
    run_stage("pretrain", augmentation_reviewed=AUGMENTATION_REVIEWED)

### Matched downstream experiments

In [111]:
METHOD_FILTER = ["simclr", "scratch", "imagenet"]
BUDGET_FILTER = [0.01]
SEED_FILTER = [0]
# Set each filter to None for the complete matrix.
RUN_DOWNSTREAM = False
if RUN_DOWNSTREAM:
    run_stage("downstream", methods=METHOD_FILTER, budgets=BUDGET_FILTER, seeds=SEED_FILTER)

### Review development results

In [112]:
RUN_DEVELOPMENT_REPORT = False
if RUN_DEVELOPMENT_REPORT:
    run_stage("aggregate", partition="development")

### Record a development-only selection

In [113]:
SELECTED_RUNS = {}  # Map completed downstream run directories to development-selection reasons.
RUN_SELECTION = False
if RUN_SELECTION:
    with project_directory():
        for run_dir, reason in SELECTED_RUNS.items():
            lock_run(Path(run_dir), reason)

### Evaluate locked runs once

In [114]:
RUN_FINAL_EVALUATE = False
if RUN_FINAL_EVALUATE:
    run_stage("evaluate", methods=METHOD_FILTER, budgets=BUDGET_FILTER, seeds=SEED_FILTER)

### Aggregate held-out metric artifacts

In [115]:
RUN_FINAL_REPORT = False
if RUN_FINAL_REPORT:
    run_stage("aggregate", partition="final_validation")

## Optional operations

`execute_task(["repair_pretrain_manifest", ...])` repairs a legacy SSL manifest without resampling cohorts. `execute_task(["analyze_errors", ...])` exports a reviewer worksheet. Use `--help` on any named task for its arguments.

VLM tasks are experimental and development-only in this consolidation. They require `.[vlm]`, a radiology-domain checkpoint pinned to an immutable revision, model-card URL, explicit license and overlap review, and `image-text VLM` reporting. No checkpoint is downloaded by default. The primary image-only evaluation lock is not a VLM final-evaluation implementation.

The test suite loads only cells tagged `library`. Workflow cells are excluded. No checkpoint, data cache, or result is embedded in this notebook.